# Descripción tablas

- cod_cdu_forma_literaria: Tabla de códigos que hace referencia a las posibles formas literarias de los documentos descritos en los registros. Información a partir del número CDU.

- cod_forma_literaria_008: Tabla de códigos que hace referencia a las posibles formas literarias de los documentos descritos en los registros. Información a partir del número campo 008 presente en el catálogo.

- cod_lengua_marc

- cod_pais_marc

- cod_publico_008: Tabla de códigos que hace referencia a los posibles públicos destinatarios de los documentos descritos en los registros. Información a partir del número campo 008 presente en el catálogo.

- cod_tipo_fecha_008: Tabla de códigos que hace referencia a la interpretación de la fecha del campo fecha_008 de los documentos descritos en los registros. Información a partir del número campo 008 presente en el catálogo.


- Ejemplares: Tabla donde se muestran códigos de ejemplar físico. Su PK es el código de barras del libro (mismo que prbarc en prestamos). Y une con codigo_id del registro. Un registro puede tener muchos ejemplares. Junto con  información característica de cada uno de ellos (campos finales):
    - biblioteca a la que pertenecen: cada ejemplar registra biblioteca asignada según nombre biblioteca (nombre distrito + nombre biblioteca) en el **campo sucursal**. -> Se deja campo completo unido, como información complementaria. Esta tabla se une con prestamos por el barcode (prbarc) y esta se une con info bibliotecas por su prprsu.
    - Localización dentro de la biblioteca
    - Signatura: valores tejuelo. El campo signatura corresponde al número de control de colocación del ejemplar, está compuesto de la clase de volumen (letras o números) + 3 letras que definen al autor + 3 letras que definen el título. En la sección de clase, se puede obtener materia, público destinatario y en el caso de las novelas juveniles, el género (más información en el documento anexo Guía en lenguaje claro).
    - Notas divididas según origen descripción. Proveniente del campo subcampo.

- Registro: Tabla donde se muestra la información específica de cada registro, hay uno por elemento presente en la red de bibliotecas nacional. Una misma obra puede tener más de un registro.


# Acciones iniciales

In [ ]:
from google.colab import drive # montar drive
import shutil # copiar docs. drive en disco local
from pathlib import Path# Gestionar rutas archivos
import json # Para crear y tratar archivo json
import pandas as pd
from IPython.display import display
import numpy as np

pd.options.mode.copy_on_write = True
pd.set_option("io.parquet.engine", "pyarrow")
pd.set_option("mode.string_storage", "pyarrow")

In [ ]:
# Montar drive:
drive.mount('/content/drive')

## Constantes


In [ ]:
# 1. Rutas originales
pf = Path('drive')
local = Path('/content')

# 2. Rutas sobre las que trabajar:
# 2.1 Rutas que apuntan hacia Drive
DRIVE_ORIGEN = pf / 'bronze'
DRIVE_DESTINO = pf / 'silver'

# 2.2 Rutas que apuntan hacia disco local:
ORIGEN_LOCAL = local / 'datos_bronze'
DESTINO_LOCAL = local / 'resultados_silver'
ORIGEN_LOCAL.mkdir(parents=True, exist_ok=True)
DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)


# 3. Escritura CSV
PARAMETROS_TO_CSV = {'sep': ";", 'index': False,
                     'encoding': "utf-8-sig", 'decimal': ","}

# 4. Limpieza y transformaciones DF
PATRONES_NULOS = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan","sd", "s/d", "sin datos",
                  "tbd", "x", "xx", "xxx")

## Copia archivos drive en local

In [ ]:
# Copiar todas las subcarpetas de origen drive
#pudiendo excluir algunas.

EXCLUIR = ["PRESTAMOS_HISTORICOS"]
assert DRIVE_ORIGEN.exists(), f"No existe {DRIVE_ORIGEN}"

disponibles = sorted(p.name for p in DRIVE_ORIGEN.iterdir() if p.is_dir())
seleccion = [s for s in disponibles if s not in EXCLUIR]

faltan = [s for s in seleccion if s not in disponibles]
assert not faltan, f"No existen en DRIVE_ORIGEN: {faltan}. Disponibles: {disponibles}"

for sub in seleccion:
    shutil.copytree(DRIVE_ORIGEN / sub, ORIGEN_LOCAL / sub, dirs_exist_ok=True)

DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)

print(f"Subcarpetas en Drive: {disponibles}")
print(f"Copiadas: {seleccion}\n")

for sub in seleccion:
    csvs = sorted(p.name for p in (ORIGEN_LOCAL / sub).rglob("*.csv"))
    print(f"{sub} ({len(csvs)} csv): {csvs}")

In [ ]:
# Importación archivo JSON desde Drive:

jsons = list(DRIVE_ORIGEN.glob('*.json'))
assert jsons, f'No hay ningún .json en {DRIVE_ORIGEN}'
for f in jsons:
    shutil.copy2(f, ORIGEN_LOCAL / f.name)
print(f"JSON copiados: {[f.name for f in jsons]}")


# Lectura archivo JSON para emplearlo en lectura CSV:

with open(ORIGEN_LOCAL / 'parametros_csv_brutos.json', encoding='utf-8') as f:
    params_json = json.load(f)

# Lectura CSV del catálogo y tablas descripción

### Lectura CSV - DIMENSIONES_CATALOGO

In [ ]:
b_dim_cat, fallos_dim_cat = lectura_bronze(
    params_json,
    'DIMENSIONES_CATALOGO_MARC',
    lista_carpetas,
    ORIGEN_LOCAL
    )

CSV correctamente leidos (6)
 Nombre y Dimensiones de los DF: 
cod_cdu_forma_literaria_ok: 26 filas y 3 columnas
cod_forma_literaria_008: 14 filas y 3 columnas
cod_lengua_marc: 516 filas y 3 columnas
cod_pais_marc: 335 filas y 4 columnas
cod_publico_008: 10 filas y 3 columnas
cod_tipo_fecha_008: 15 filas y 3 columnas


### Lectura CSV - CATALOGO_MARC

Registros y ejemplares

In [ ]:
b_catalogo, fallos_catalogo = lectura_bronze(
    params_json,
    'CATALOGO_MARC',
    lista_carpetas,
    ORIGEN_CSV
    )

CSV correctamente leidos (2)
 Nombre y Dimensiones de los DF: 
ejemplares_v4: 3619006 filas y 8 columnas
registros_v4: 656404 filas y 41 columnas


# Limpiezas comunes

### Limpieza CSV - DIMENSIONES_CATALOGO_MARC

In [ ]:
dfs_dim_catalogo, resumen_nulos_dim_c = transformaciones_comunes(b_dim_cat)


--------------- DF: cod_cdu_forma_literaria_ok ---------------

1. Información incial:

1.1 Muestra:



,codigo,descripcion,verificado
15,90,Otras formas literarias (82-9),heredado del padre



1.2 Información general:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   codigo       26 non-null     string
 1   descripcion  26 non-null     string
 2   verificado   26 non-null     string
dtypes: string(3)
memory usage: 756.0 bytes

1.3 Descripción inicial:



,codigo,descripcion,verificado
count,26,26,26
unique,26,20,5
top,1,Otras formas literarias (82-9),CDU
freq,1,6,13



2. Resultados post normalizaciones del DF cod_cdu_forma_literaria_ok:
2.1 Muestra:



,codigo,descripcion,verificado
22,97,otras formas literarias (82-9),heredado del padre


... (41 outputs left out)

,codigo,descripcion,fuente
10,r,reimpresion/reedicion (fecha1 = reproduccion; ...,"library of congress, marc 21 format for biblio..."



2.2 Descripción:



,codigo,descripcion,fuente
count,15,15,15
unique,15,15,1
top,b,sin fechas; fecha a. c.,"library of congress, marc 21 format for biblio..."
freq,1,1,15



3. Primera inspección posibles nulos de la tabla cod_tipo_fecha_008:

La tabla cod_tipo_fecha_008 no presenta posibles valores nulos.


### Limpieza CSV - CATALOGO_MARC

In [ ]:
cat_dfs, cat_resumen_nulos = transformaciones_comunes(b_catalogo)


--------------- DF: ejemplares_v4 ---------------

1. Información incial:

1.1 Muestra:



,barcode_norm,barcode_original,id_registro,biblioteca_852a,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
2090525,101237536,101237536,ES-MaRBCM01530360c-9,MADRID. Bibliotecas Públicas,USERA. BP Gabriel García Márquez,Sala Infantil,VID INF ACA,



1.2 Información general:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3619006 entries, 0 to 3619005
Data columns (total 8 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   barcode_norm       string
 1   barcode_original   string
 2   id_registro        string
 3   biblioteca_852a    string
 4   sucursal_852b      string
 5   localizacion_852c  string
 6   signatura_852j     string
 7   subcampo_852m      string
dtypes: string(8)
memory usage: 220.9 MB

1.3 Descripción inicial:



,barcode_norm,barcode_original,id_registro,biblioteca_852a,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
count,3619006,3619006,3619006,3619006,3619006,3619006,3619006,3619006
unique,3618957,3618957,619957,1,63,40,717962,87418
top,1401877988,1401877988,ES-MaRBCM01011307c-5,MADRID. Bibliotecas Públicas,SALAMANCA. BPE Manuel Alvar,Sala adultos,,
freq,2,2,1970,3619006,420297,1907186,49599,2740497



2. Resultados post normalizaciones del DF ejemplares_v4:
2.1 Muestra:



,barcode_norm,barcode_original,id_registro,biblioteca_852a,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
2046266,101791327,101791327,es-marbcm01396185c-0,madrid. bibliotecas publicas,centro. bp ivan de vargas,sala adultos,cdm roc bjo,


... (10 outputs left out)


2.2 Descripción:



,id_registro,ld06,ld07,es_libro,len_008,tipo_fecha_008,fecha1_008,fecha2_008,pais_edicion_008,lengua_texto_008,...,genero_literario_cdu,lengua_literatura_cdu,lugar_literatura_cdu,critica_literaria_cdu,materias_650,generos_655,tipo_publico_521,publico_destinatario_521a,nota_publico_521,n_ejemplares
count,656404,656404,656404,656404,656404,656404,656404,656404,656404,656404,...,656404,656404,656404,656404,656404,656404,656404,656404,656404,656404
unique,656402,19,16,2,38,37,424,430,388,225,...,819,566,447,2,90704,13471,14,3386,3397,332
top,es-marbcm02153130c-0,a,m,1,40,s,2007,####,sp,spa,...,,,,,,,,,,1
freq,2,519754,651168,515374,572248,621876,26615,646994,583600,554307,...,443114,438697,596978,650961,394312,391317,558445,558466,558446,299745



3. Primera inspección posibles nulos de la tabla registros_v4:


Resumen posibles nulos encontrado:



columna,anio_publicacion,autor,cdu_080,critica_literaria_cdu,fecha1_008,fecha2_008,forma_literaria_008,forma_literaria_cdu,genero_literario_cdu,generos_655,...,subtitulo_245b,tipo_fecha_008,tipo_publico_521,titulo_245a,titulo_orden_245a,titulo_original_nota,titulo_uniforme_130,titulo_uniforme_240,traduccion_041_ind1,total
'',41455,163407,108276,650961,411,1090,1937,443114,443114,391317,...,455331,398,558445,93,158,654893,634104,631529,589909,11638213
'-',0,0,0,0,0,0,0,0,0,0,...,0,4,0,0,0,0,0,0,0,8
'.',0,0,0,0,0,0,131,0,0,0,...,0,0,0,0,2,0,0,0,0,133
'?',0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
'na',0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,2
'x',0,0,0,0,0,0,0,0,0,0,...,0,0,0,9,8,0,0,0,0,20
'xx',0,0,0,0,0,0,0,0,0,0,...,0,0,0,3,3,0,0,0,0,1666
'xxx',0,0,0,0,0,0,0,0,0,0,...,0,0,0,15,15,0,0,0,0,480
total,41455,163407,108276,650961,411,1090,2068,443114,443114,391317,...,455331,403,558445,120,188,654893,634104,631529,589909,11640523


# Limpiezas particulares por DF - DIMENSIONES CATÁLOGO

Se revisan valores que estén correctos pero se añadirán las descripciones de los distintos códigos a la tabla registros del Catálogo Marc.

In [ ]:
DESTINO_DIMENSIONES_CAT = DRIVE_DESTINO / 'DIMENSIONES_CATALOGO'
DESTINO_DIMENSIONES_CAT.mkdir(parents=True, exist_ok=True)

In [ ]:
dfs_dim_catalogo.keys()

dict_keys(['cod_cdu_forma_literaria_ok', 'cod_forma_literaria_008', 'cod_lengua_marc', 'cod_pais_marc', 'cod_publico_008', 'cod_tipo_fecha_008'])

## 1.cod_cdu_forma_literaria

Código que hace referencia a las posibles formas literarias de los documentos descritos en los registros. Información obtenida a partir del número CDU presente en el catálogo.

Se observa que todos los valores para los campos código y descripción son únicos y no se observan posibles valores nulos en ninguno de los dos campos. Los otros dos (verificado y fuente) es información no necesaria, determina origen de la información. Se observan valores nulos.

In [ ]:
# Se asigna variable al df
dc = "cod_cdu_forma_literaria_ok"
df_dc1 = dfs_dim_catalogo[dc]

In [ ]:
# Se ven posibles nulos:
display(resumen_nulos_dim_c[dc])

None

In [ ]:
# Se miran valores duplicados en descripción
descripcion_duplicados = df_dc1['descripcion'].duplicated(keep=False)
df_dc1[descripcion_duplicados]

,codigo,descripcion,verificado
12,83,miscelaneas (82-8),heredado del padre
13,85,miscelaneas (82-8),heredado del padre
15,90,otras formas literarias (82-9),heredado del padre
20,95,otras formas literarias (82-9),heredado del padre
21,96,otras formas literarias (82-9),heredado del padre
22,97,otras formas literarias (82-9),heredado del padre
23,98,otras formas literarias (82-9),heredado del padre
24,99,otras formas literarias (82-9),heredado del padre


Las duplicidades en descripciones corresponden a que estos códigos describen los mismos parámetros acorde a la disposición y valor de los valores dentro de la estructura del número CDU.

### Asignar nulos explícitos

In [ ]:
columnas_df = list(df_dc1.columns)
df_dc1[columnas_df] = df_dc1[columnas_df].replace(
    PATRONES_NULOS, pd.NA
    )

### Últimas observaciones

In [ ]:
df_dc1.info()
display(df_dc1.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   codigo       26 non-null     string
 1   descripcion  26 non-null     string
 2   verificado   26 non-null     string
dtypes: string(3)
memory usage: 756.0 bytes


,codigo,descripcion,verificado
count,26,26,26
unique,26,20,5
top,1,otras formas literarias (82-9),cdu
freq,1,6,13


### Exportar CSV
Se da por buena la tabla cod_cdu_forma_literaria. Se procede a exportar en csv, bajo el mismo nombre. Por si hace falta

In [ ]:
dict(df_dc1.dtypes)

{'codigo': string[python],
 'descripcion': string[python],
 'verificado': string[python]}

In [ ]:
df_dc1.to_csv(DESTINO_DIMENSIONES_CAT / f"{dc}.csv", **PARAMETROS_TO_CSV)


## 2.cod_forma_literaria_008

Código que hace referencia a la información obtenida a partir del campo 008 presente en el catálogo.

Señala distintas categorías contempladas dentro de formas literarias

Se observa que todos los valores para los campos código y descripción son únicos y no se observan posibles valores nulos.

In [ ]:
dc = 'cod_forma_literaria_008'
df_dc2 = dfs_dim_catalogo[dc]

### Asignar nulos explícitos

In [ ]:
# Se ven posibles nulos:
display(resumen_nulos_dim_c[dc])

None

In [ ]:
columnas_df = list(df_dc2.columns)
df_dc2[columnas_df] = df_dc2[columnas_df].replace(
    PATRONES_NULOS, pd.NA
    )

### Últimas observaciones

In [ ]:
df_dc2.info()
display(df_dc2.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   codigo       14 non-null     string
 1   descripcion  14 non-null     string
 2   fuente       14 non-null     string
dtypes: string(3)
memory usage: 468.0 bytes


,codigo,descripcion,fuente
count,14,14,14
unique,14,14,1
top,0,no ficcion,"library of congress, marc 21 format for biblio..."
freq,1,1,14


### Exportar CSV
Se da por buena la tabla.

In [ ]:
columnas_df
dict(df_dc2.dtypes)

{'codigo': string[python],
 'descripcion': string[python],
 'fuente': string[python]}

In [ ]:
df_dc2.to_csv(DESTINO_DIMENSIONES_CAT / f"{dc}.csv",**PARAMETROS_TO_CSV)


## 3.cod_lengua_marc

Se observa valor duplicado para el campo nombre_en. Sin posibles valores nulos.

In [ ]:
dc = 'cod_lengua_marc'
df_dc3 = dfs_dim_catalogo[dc]

In [ ]:
# Se ven posibles nulos:
display(resumen_nulos_dim_c[dc])

None

In [ ]:
# Asignación nulos explícitos
columnas_df = list(df_dc3.columns)
df_dc3[columnas_df] = df_dc3[columnas_df].replace(
    PATRONES_NULOS, pd.NA
    )

In [ ]:
# Se miran valores duplicados en nombre_en:
dupli_nombre_en = df_dc3[df_dc3.duplicated(subset="nombre_en", keep=False)]
print(f"Total registros duplicados : {len(dupli_nombre_en)}")
display(dupli_nombre_en.head(6))

Total registros duplicados : 48


,codigo,nombre_en,obsoleto
99,hrv,croatian,no
100,scr,croatian,si
128,epo,esperanto,no
129,esp,esperanto,si
131,gez,ethiopic,no
132,eth,ethiopic,si


Se ve que hay duplicidad de códigos entre códigos obsoletos o no.

Se procede a eliminar aquellos registros donde el código está obsoleto (si).

In [ ]:
# Se hace filtro sobre los registros seleccionados a eliminar:
eliminar = dupli_nombre_en.loc[dupli_nombre_en['obsoleto'] == 'si']

# Se eliminan:
df_dc3 = df_dc3.drop(index=eliminar.index).reset_index(drop=True)

In [ ]:
# Comprobación de valores duplicados:

print(f"Total registros duplicados : {df_dc3.duplicated().sum()}")

Total registros duplicados : 0


### Últimas observaciones

In [ ]:
df_dc3.info()
display(df_dc3.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 492 entries, 0 to 491
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   codigo     492 non-null    string
 1   nombre_en  492 non-null    string
 2   obsoleto   492 non-null    string
dtypes: string(3)
memory usage: 11.7 KB


,codigo,nombre_en,obsoleto
count,492,492,492
unique,492,492,2
top,abk,abkhaz,no
freq,1,1,485


### Exportar CSV
Se da por buena la tabla.

In [ ]:
columnas_df
dict(df_dc3.dtypes)

{'codigo': string[python],
 'nombre_en': string[python],
 'obsoleto': string[python]}

In [ ]:
df_dc3.to_csv(DESTINO_DIMENSIONES_CAT / f"{dc}.csv",**PARAMETROS_TO_CSV)


## 4.cod_pais_marc
Se observan valores duplicados para el campo codigo y nombre_en y posibles valores nulos en el campo codigo.

In [ ]:
dc = 'cod_pais_marc'
df_dc4 = dfs_dim_catalogo[dc]

In [ ]:
# Se ven posibles nulos:
print("Posibles valores nulos:")
display(resumen_nulos_dim_c[dc])

# Tamaño total df
print(f"Tamaño total df {df_dc4.shape}")

Posibles valores nulos:


columna,codigo,total
'sd',1,1
'xx',1,1
total,2,2


Tamaño total df (335, 4)


Se han observado dos posibles valores nulos en el campo codigo. Se ven registros

In [ ]:
df_dc4[df_dc4['codigo'].isin(PATRONES_NULOS)]

,codigo,nombre_en,region,obsoleto
208,xx,"no place, unknown, or undetermined",other,no
275,sd,south sudan,africa,no


El código sd es legítimo, pertenece a Sudán del Sur. El otro pertenece a un código genérico para determinar lugares no conocidos. Ambos se dan por válidos.

In [ ]:
# Se asignan valores nulos explícitos
# Extrayendo valores observados de la lista
posibles_nulos = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan", "s/d", "sin datos",
                  "tbd", "x", "xxx")

# Asignación nulos explícitos
columnas_df = list(df_dc4.columns)
df_dc4[columnas_df] = df_dc4[columnas_df].replace(
    posibles_nulos, pd.NA
    )

# Comprobación:
print(f"Nulos en df: {df_dc4.isna().sum().sum()}")

Nulos en df: 0


In [ ]:
#Se procede a ver valores duplicados:

print("Cantidad duplicados completos: \n")
display(df_dc4.duplicated(keep=False).sum())

#en código:
dupli_codigo = df_dc4[df_dc4.duplicated(subset='codigo', keep=False)]
#En nombre:
dupli_nombre = df_dc4[df_dc4.duplicated(subset='nombre_en', keep=False)]

print(f"\nHay {len(dupli_codigo)} registros duplicados según clave código: \n")
display(dupli_codigo)


print(f"\nHay {len(dupli_nombre)} registros duplicados según clave nombre: \n")
display(dupli_nombre)

Cantidad duplicados completos: 



np.int64(0)


Hay 2 registros duplicados segun calve codigo: 



,codigo,nombre_en,region,obsoleto
10,ai,anguilla,west indies,si
16,ai,armenia (republic),asia,no



Hay 2 registros duplicados segun calve codigo: 



,codigo,nombre_en,region,obsoleto
9,am,anguilla,west indies,no
10,ai,anguilla,west indies,si


Valor código duplicado con nombre distinto un registro obsoleto y el otro no. Se elimina el obsoleto.

El nombre del código duplicado y obsoleto pertenece al nombre duplicado.

In [ ]:
# Eliminación registro código duplicado y obsoleto
print(f"Registros antes de eliminar duplicados: {len(df_dc4)}")
eliminar_cod = dupli_codigo.loc[dupli_codigo['obsoleto'] == 'si']
df_dc4 = df_dc4.drop(index=eliminar_cod.index).reset_index(drop=True)

# Comprobación:
print(f"\nRegistros despues de eliminar duplicados: {len(df_dc4)}")

Registros antes de eliminar duplicados: 335

Registros despues de eliminar duplicados: 334


### Últimas observaciones

In [ ]:
df_dc4.info()
display(df_dc4.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   codigo     334 non-null    string
 1   nombre_en  334 non-null    string
 2   region     334 non-null    string
 3   obsoleto   334 non-null    string
dtypes: string(4)
memory usage: 10.6 KB


,codigo,nombre_en,region,obsoleto
count,334,334,334,334
unique,334,334,12,1
top,af,afghanistan,north america,no
freq,1,1,69,334


### Exportar CSV
Se da por buena la tabla.

In [ ]:
columnas_df
dict(df_dc4.dtypes)

{'codigo': string[python],
 'nombre_en': string[python],
 'region': string[python],
 'obsoleto': string[python]}

In [ ]:
df_dc4.to_csv(DESTINO_DIMENSIONES_CAT / f"{dc}.csv",**PARAMETROS_TO_CSV)


## 5.cod_publico_008

Tabla correcta, sin duplicados ni nulos. Se procede a exportar como CSVs.

In [ ]:
dc = 'cod_publico_008'
df_dc5 = dfs_dim_catalogo[dc]

In [ ]:
# Se ven posibles nulos:
display(resumen_nulos_dim_c[dc])

None

### Asignar nulos explícitos

In [ ]:
columnas_df = list(df_dc5.columns)
df_dc5[columnas_df] = df_dc5[columnas_df].replace(
    PATRONES_NULOS, pd.NA
    )

### Últimas observaciones

In [ ]:
df_dc5.info()
display(df_dc5.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   codigo       10 non-null     string
 1   descripcion  10 non-null     string
 2   fuente       10 non-null     string
dtypes: string(3)
memory usage: 372.0 bytes


,codigo,descripcion,fuente
count,10,10,10
unique,10,10,1
top,#,desconocido o no especificado,"library of congress, marc 21 format for biblio..."
freq,1,1,10


### Exportar CSV
Se da por buena la tabla.

In [ ]:
columnas_df
dict(df_dc5.dtypes)

{'codigo': string[python],
 'descripcion': string[python],
 'fuente': string[python]}

In [ ]:
df_dc5.to_csv(DESTINO_DIMENSIONES_CAT / f"{dc}.csv",**PARAMETROS_TO_CSV)


## 6.cod_tipo_fecha_008

In [ ]:
dc = 'cod_tipo_fecha_008'
df_dc6 = dfs_dim_catalogo[dc]

In [ ]:
# Se ven posibles nulos:
display(resumen_nulos_dim_c[dc])

None

### Asignar nulos explícitos

In [ ]:
columnas_df = list(df_dc6.columns)
df_dc6[columnas_df] = df_dc6[columnas_df].replace(
    PATRONES_NULOS, pd.NA
    )

### Últimas observaciones

In [ ]:
df_dc6.info()
display(df_dc6.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   codigo       15 non-null     string
 1   descripcion  15 non-null     string
 2   fuente       15 non-null     string
dtypes: string(3)
memory usage: 492.0 bytes


,codigo,descripcion,fuente
count,15,15,15
unique,15,15,1
top,b,sin fechas; fecha a. c.,"library of congress, marc 21 format for biblio..."
freq,1,1,15


### Exportar CSV
Se da por buena la tabla.

In [ ]:
columnas_df
dict(df_dc6.dtypes)

{'codigo': string[python],
 'descripcion': string[python],
 'fuente': string[python]}

In [ ]:
df_dc6.to_csv(DESTINO_DIMENSIONES_CAT / f"{dc}.csv",**PARAMETROS_TO_CSV)

# Ejemplares - Limpiezas particulares

Tabla donde se muestran los datos por ejemplar contemplados dentro del catálogo marc.



In [ ]:
# Se determina nombre de búsqueda df:
nombre = "ejemplares_v4"
# Se extrae df ejemplares del diccionario del catálogo:
df_ej = cat_dfs[nombre]

In [ ]:
# Valores iniciales antes de las transformaciones:
print(f"Las dimensiones iniciales del df son de {df_ej.shape}."
f"\nLos valores nulos totales son: {cat_resumen_nulos[nombre].loc['total', 'total']}"
"\nRepartidos como se muestra en la siguiente tabla:\n")

# Resumen nulos
display(cat_resumen_nulos[nombre])

Las dimensiones inciales del df son de (3619006, 8).
Los valores nulos totales son: 2790113
Repartidos como se muestra en la siguiente tabla:



columna,localizacion_852c,signatura_852j,subcampo_852m,total
'',1,49599,2740497,2790097
'x',0,0,16,16
total,1,49599,2740513,2790113


### Duplicados completos

In [ ]:
# DUPLICADOS COMPLETOS
# todos los registros duplicados:
duplis_completos = df_ej[df_ej.duplicated(keep=False)]

# Se mira cantidad de registros completamente duplicados:
print(f"Hay {len(duplis_completos)} registros totalmente duplicados")

# Se mira cantidad de grupos de registros completamente duplicados:

tam = duplis_completos.value_counts(subset='barcode_original',dropna=False) # Duplicados teniendo en cuenta nulos
n_grupos = len(tam)
print(f"Numero de grupos registros duplicados: {len(tam)}"
"\nLos grupos se distribuyen de la siguiente manera: \n")
display(tam.value_counts())
ej_eliminar = tam.sum() - len(tam)
print(f"\nSe eliminaran {ej_eliminar} registros."
f"Dejando {len(df_ej) - ej_eliminar} registros restantes")


Hay 98 registros totalmente duplicados
Numero de grupos registros duplicados: 49
Los grupos se distribuyen de la siguiente manera: 



,count
count,
2,49



Se eliminaran 49 registros.Dejando 3618957 registros restantes


Se han encontrado 49 grupos de registros completamente duplicados. Dos registros por cada grupo, 98 registros en total. Se procede a eliminar el segundo ya que no hay relevancia entre uno y otro.

In [ ]:
# Eliminación de uno de los registros:
print(f"filas iniciales:{len(df_ej)}")
df_ej = df_ej.drop_duplicates()
print(f"Num reg eliminados {ej_eliminar}")
ej_finales = len(df_ej)
print(f"Reg finales totales {ej_finales}")

filas iniciales:3619006
Num reg eliminados 49
Reg finales totales 3618957


Las dimensiones iniciales del df son de (3619006, 8).

Los valores nulos no explícitos totales son 2790113 repartidos en los campos localizacion_852c (1), signatura_852j (49.599) y subcampo_852m (2.740.513)

filas después de eliminación duplicados completos: 3.618.957

### Duplicados parciales

Por clave = cod. barras

In [ ]:
# Duplicados en código de barras:
clave  = 'barcode_original'
duplis_clave = df_ej[df_ej.duplicated(subset=clave, keep=False)]
n_reg_duplis_calve = len(duplis_clave)
if n_reg_duplis_calve == 0:
  print("No hay registros duplicados según la clave")
else:
  tam2 = duplis_clave.value_counts(subset='barcode_original',dropna=False)
  print(f"Se han encontrado los {n_reg_duplis_calve}"
        f"con la clave - {clave} duplicada")
  display(tam2)

del duplis_completos, tam, n_grupos, ej_eliminar, duplis_clave, n_reg_duplis_calve

No hay registros duplicados según la calve


### Eliminación campos

No útiles para el análisis:
- barcode_normalizado: Normalizo desde el original
- biblioteca_852a: Describe la red de bibliotecas a la que pertenece.

Se mantienen:
- sucursal: Describe la biblioteca específica que contiene el ejemplar.
- localización: Donde se encuentra el libro dentro de la biblioteca.
- Signatura: tejuelo
- Subcampo: Información de campo abierto.

In [ ]:
print(f"Numero de campos antes de eliminar campos seleccionados: {df_ej.shape[1]}")

campos_eliminar = ["barcode_norm", "biblioteca_852a"]
# Se eliminan columnas:
df_ej = df_ej.drop(columns=campos_eliminar)

print(f"\nNumero de campos antes de eliminar campos seleccionados: {df_ej.shape[1]}")

Numero de campos antes de eliminar campos seleccionados: 8

Numero de campos antes de eliminar campos seleccionados: 6


### Comprobación nulos y asignación valor nulo explícito

In [ ]:
# Se vuelve a mirar cantidad de nulos:
resumen = buscar_nulos(df_ej)
display(resumen)
# Asignar nueva cantidad de posibles nulos encontradas:
total_pos_nulos2 = resumen.loc['total', 'total']

columna,localizacion_852c,signatura_852j,subcampo_852m,total
'',1,49599,2740450,2790050
'x',0,0,16,16
total,1,49599,2740466,2790066


Siguen apareciendo nulos en los campos de descripción de los ejemplares en cuanto a información relativa a la ubicación del mismo en la biblioteca o información genérica documentada en el campo abierto subcampo_852m.

Se procede a ver si estos registros (barcode) presentan información trasladable en otros, según el id_registro. Un mismo id_registro puede estar presente en más de un barcode.

Se valoran primero los posibles nulos con valor x, luego los de valor vacío en campo localización y signatura, que son los campos en que se puede asignar info por relación. Al campo subcampo, al ser abierto, no se le puede asignar valor propio, se revisa pero sin intención de asignar valores.

**POSIBLE VALOR NULO X**

In [ ]:
# Se filtran registros cuyo valor en subcampo es = x
nulos_x = df_ej[df_ej['subcampo_852m'] == 'x']
# Número de registros
print(f"Hay {len(nulos_x)} registros con una x marcada en su subcampo.\n")
display(nulos_x.head(1))
# Se comprueba que todos los registros son del mismo centro.
nulos_x.value_counts(subset=['sucursal_852b', 'localizacion_852c', 'subcampo_852m'])

Hay 16 registros con una x marcada en su subcampo.



,barcode_original,id_registro,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
617298,100289261,es-marbcm00369415c-3,salamanca. bp david gistau,deposito,b gar,x


,,,count
sucursal_852b,localizacion_852c,subcampo_852m,
salamanca. bp david gistau,deposito,x,16


Los registros con valor x en subcampo pertenecen al mismo centro, parece ser forma que tienen de determinar ciertos registros ubicados en depósito.

Se mantienen como están.

**POSIBLE VALOR NULO EN VALOR VACIO("") DEL CAMPO LOCALIZACION**

In [ ]:
# Se filtran registros cuyo valor en subcampo localizacion es = ""
nulos_vacio_loc = df_ej[df_ej['localizacion_852c'] == '']
# Número de registros
print(f"Hay {len(nulos_vacio_loc)} registros con valor vacio en localizacion.\n")
display(nulos_vacio_loc.head(1))


Hay 1 registros con valor vacio en localizacion.



,barcode_original,id_registro,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
2332912,1100106834,es-marbcm01479064c-4,latina. bp antonio mingote,,,


In [ ]:
# Se revisan más registros con el mismo id_registro y sucursal:
cols = ["id_registro", "sucursal_852b"]

mask = pd.MultiIndex.from_frame(df_ej[cols]).isin(
    pd.MultiIndex.from_frame(nulos_vacio_loc[cols])
)
print("registros con mismo id_registro y sucursal")
display(df_ej[mask].sort_values(cols))

# Se miran registros con mismo id_registro:
mask_id = df_ej["id_registro"].isin(nulos_vacio_loc["id_registro"])
print("\nregistros con mismo id_registro.")
display(df_ej[mask].sort_values("id_registro"))

registros con mismo id_registro y sucursal


,barcode_original,id_registro,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
2332912,1100106834,es-marbcm01479064c-4,latina. bp antonio mingote,,,



registros con mismo id_registro.


,barcode_original,id_registro,sucursal_852b,localizacion_852c,signatura_852j,subcampo_852m
2332912,1100106834,es-marbcm01479064c-4,latina. bp antonio mingote,,,


No hay más ejemplares con ese número de registro en todo el DF, no se puede asignar valor al mismo. No se pasa como nulo, ya que la falta de información es importante. Se asigna valor no_documentado.

In [ ]:
# Se pasa a no documentado:
print(f" Hay {len(df_ej[df_ej['localizacion_852c'] == ''])} valores vacios "
"antes de reasignación en este campo")
df_ej['localizacion_852c'] = df_ej['localizacion_852c'].replace("", "no_documentado")
print(f" Hay {len(df_ej[df_ej['localizacion_852c'] == ''])} valores vacios "
"despues de la asignación en este campo")

 Hay 1 valores vacios antes de reasignación en este campo
 Hay 0 valores vacios despues de la asignación en este campo


**POSIBLE VALOR NULO EN VALOR VACIO("") DEL CAMPO SIGNATURA**

In [ ]:
# Revisión composición de los valores vacíos en los campos:

# 1. SIGNATURA:
# Se filtran registros cuyo valor ""
nulos_vacio_sig = df_ej[df_ej['signatura_852j'] == '']

# 1.1 Cantidad de registros (id_registro)
# Grupos de ejemplares según id_registro
id_reg_vacio_sig = nulos_vacio_sig['id_registro'].unique()
# Grupo de ejemplares con el mismo id y en la misma biblioteca
id_bib_sig = nulos_vacio_sig.value_counts(subset=['id_registro',"sucursal_852b"])

# Número de registros
print(f"------ SIGNATURA------\n"
f"\nHay {len(nulos_vacio_sig)} registros con valor vacio en el "
"campo signatura.")
print(f"\nEstos ejemplares corresponden {len(id_reg_vacio_sig)} id_registro unicos:"
f"\nDe los cuales, {len(id_bib_sig)} ids_registro comparten biblioteca."
"\nEntendiendose que un hay más de un libro dentro de la misma biblioteca.")

# 2. SUBCAMPO
# Se filtran registros cuyo valor ""
nulos_vacio_sub = df_ej[df_ej['subcampo_852m'] == '']

# 1.1 Cantidad de registros (id_registro)
# Grupos de ejemplares según id_registro
id_reg_vacio_sub = nulos_vacio_sub['id_registro'].unique()
# Grupo de ejemplares con el mismo id y en la misma biblioteca
id_bib_sub = nulos_vacio_sub.value_counts(subset=['id_registro',"sucursal_852b"])

# Número de registros
print(f"\n\n------ SUBCAMPO------\n"
f"\nHay {len(nulos_vacio_sub)} registros con valor vacio en el "
"campo signatura.")
print(f"\nEstos ejemplares corresponden {len(id_reg_vacio_sub)} id_registro unicos:"
f"\nDe los cuales, {len(id_bib_sub)} ids_registro comparten biblioteca."
"\nEntendiendose que hay más de un libro dentro de la misma biblioteca.")

------ SIGNATURA------
Hay 49599 registros con valor vacio en el campo signatura.

Estos ejemplares corresponden 1779 id_registro unicos:
De los cuales, 4735 ids_registro comparten biblioteca.
Entendiendose que un hay más de un libro dentro de la misma biblioteca.


------ SUBCAMPO------
Hay 2740450 registros con valor vacio en el campo signatura.

Estos ejemplares corresponden 546498 id_registro unicos:
De los cuales, 2493365 ids_registro comparten biblioteca.
Entendiendose que un hay más de un libro dentro de la misma biblioteca.


Se observa un mayor volumen de registros vacíos en subcampo respecto a signatura, comportamiento esperado ya que este campo corresponde a campo abierto de información.

De igual manera, se procede a buscar ejemplares que tengan el mismo id y con valores declarados para ambos campos. De encontrarse, se realizará asignación de valores.

Para signatura, se buscarán aquellos ejemplares que compartan id y tengan un único valor rellenado por grupo. No se amplía clave de selección ya que los ejemplares objeto de análisis son aquellos de origen literario/literatura y en cuyos tejuelos (valor expresado en la signatura), muestran de manera genérica la forma literaria, título y autor.

En cuanto al campo subcampo se puede observar que es un campo con tipo de información diversa, antes de asignar bajo el mismo criterio que signatura, se realiza una separación entre las distintas categorías.

Para ello procedo a pasar como nulo explícito todos estos valores vacíos encontrados. Se agruparán registros por id_registro y sucursal y se buscarán registros con valores entre los repetidos.

In [ ]:
# Asignación valores nulos explícitos:
total_vacios = resumen.iloc[0, 1] + resumen.iloc[0, 2]
print(f"Total valores vacios a convertir en nulo: {total_vacios}")
df_ej = df_ej.replace("", pd.NA)
print(f"\nTotal valores nulos post conversion: {df_ej.isna().sum().sum()}")

Total valores vacios a convertir en nulo: 2790049

Total valores nulos post conversion: 2790049


Las dimensiones iniciales del df son de (3619006, 8).

Los valores nulos no explícitos totales son 2790113 repartidos en los campos localizacion_852c (1), signatura_852j (49.599) y subcampo_852m (2.740.513)

filas después de eliminación duplicados completos: 3.618.957

Valores nulos explícitos: 2.790.049

### Asignación valores signatura y subcampo

**Acciones iniciales**

In [ ]:
# Se guarda copia si necesaria
df_or = df_ej.copy()

In [ ]:
nulos_pre_sig = df_or['signatura_852j'].isna().sum()
nulos_pre_sub = df_or['subcampo_852m'].isna().sum()
print(f"Valores nulos iniciales en signatura: {nulos_pre_sig}")
print(f"Valores nulos iniciales en subcampo: {nulos_pre_sub}")

Valores nulos iniciales en signatura: 49599
Valores nulos iniciales en subcampo: 2740450


In [ ]:

def rellenar_por_grupo(df, campo, claves, donantes=None, receptores=None):
    """Rellena nulos de `campo` con el único valor no nulo de su grupo.
    Grupos con varios valores distintos (conflicto) no se tocan.
    donantes:   máscara opcional; solo esas filas pueden ceder su valor.
    receptores: máscara opcional; solo esas filas pueden recibir valor."""
    ok = df[claves].notna().all(axis=1)
    if receptores is not None:
        ok = ok & receptores
    keys = [df[k] for k in claves]
    valores = df[campo] if donantes is None else df[campo].where(donantes)
    g = valores.groupby(keys)                       # claves NA quedan fuera

    validos   = g.transform("count")
    distintos = g.transform("nunique")
    primero   = g.transform("first")                # salta nulos

    vacio      = df[campo].isna()
    rellenable = ok & vacio & validos.gt(0)
    seguro     = rellenable & distintos.eq(1)
    conflicto  = rellenable & distintos.gt(1)

    out = df.copy()
    out.loc[seguro, campo] = primero[seguro]
    out[f"{campo}_imputado"] = seguro

    previo = df[campo].notna()
    assert out.loc[previo, campo].eq(df.loc[previo, campo]).all(), "se sobrescribió un valor"
    assert out[campo].isna().sum() == vacio.sum() - seguro.sum()

    informe = pd.Series({
        "vacíos": vacio.sum(), "rellenables": rellenable.sum(),
        "rellenados": seguro.sum(), "en conflicto": conflicto.sum(),
        "siguen vacíos": out[campo].isna().sum()})
    grupo_conf = (conflicto.groupby(keys).transform("any")
                    .reindex(df.index).fillna(False).astype(bool))
    return out, informe, df.loc[grupo_conf]


def rellenar_en_cascada(df, campo, niveles, donantes=None, receptores=None):
    """niveles: lista de (nombre, claves), del más local al más general.
    Añade `<campo>_imputacion` con el nivel del que procede cada valor."""
    out = df.copy()
    traza = f"{campo}_imputacion"
    out[traza] = pd.Series(pd.NA, index=out.index, dtype="string")
    informes, conf = {}, None

    for nombre, claves in niveles:
        relleno, inf, conf = rellenar_por_grupo(out, campo, claves, donantes, receptores)
        nuevos = relleno[f"{campo}_imputado"]
        out[campo] = relleno[campo]
        out.loc[nuevos, traza] = nombre
        informes[nombre] = inf

    return out, pd.DataFrame(informes), conf     # conf: conflictos del último nivel


def categorias(df, campo, claves):
    """Clasifica los grupos con ≥2 ejemplares según el estado de `campo`."""
    agg = (df.dropna(subset=claves).groupby(claves)[campo]
             .agg(n="size", validos="count", distintos="nunique"))
    multi = agg[agg["n"] >= 2]
    return pd.Series("otro", index=multi.index).case_when([
        (multi["validos"].eq(0),          "todos vacíos"),
        (multi["distintos"].gt(1),        "varía"),
        (multi["validos"].eq(multi["n"]), "todos iguales"),
        (multi["validos"].lt(multi["n"]), "mezcla valor + vacío"),
    ]).value_counts()


### Asignación valores nulos en signatura

In [ ]:
# Signatura: misma signatura en cada biblio?

full = df_ej.dropna(subset=["id_registro", "sucursal_852b", "signatura_852j"])
agg = full.groupby("id_registro").agg(n_bib=("sucursal_852b", "nunique"),
                                      n_sig=("signatura_852j", "nunique"))
multi = agg[agg["n_bib"] >= 2]
print(len(multi), "títulos en ≥2 bibliotecas; con signatura distinta:",
      round((multi["n_sig"] > 1).mean(), 3))

ids = multi.index[multi["n_sig"] > 1][:10]
(full[full["id_registro"].isin(ids)]
   .drop_duplicates(["id_registro", "sucursal_852b", "signatura_852j"])
   .sort_values("id_registro")
   [["id_registro", "sucursal_852b", "localizacion_852c", "signatura_852j"]])


294791 títulos en ≥2 bibliotecas; con signatura distinta: 0.596


,id_registro,sucursal_852b,localizacion_852c,signatura_852j
63918,es-marbcm00000021c-8,chamberi. bp jose luis sampedro,sala adultos,b ort gra
63919,es-marbcm00000021c-8,hortaleza. bp hortaleza,sala adultos,b ort gra
63920,es-marbcm00000021c-8,latina. bp antonio mingote,sala adultos,b ort gra
63921,es-marbcm00000021c-8,moratalaz. bp javier marias,sala adultos,b ort gra
63922,es-marbcm00000021c-8,usera. bp jose hierro,fondo local. madrid,m b ort gra
...,...,...,...,...
64042,es-marbcm00000082c-3,tetuan. bp maria zambrano,sala infantil,ave req
64043,es-marbcm00000082c-3,fuencarral-el pardo. bp rafael alberti,sala infantil,j-n req pri
64045,es-marbcm00000082c-3,puente de vallecas. bp miguel hernandez,sala infantil,j-n req pri
64046,es-marbcm00000083c-0,hortaleza. bp hortaleza,fondo local. madrid,m 394 rit


In [ ]:

PATRON_NUMERUS = r"^(?:[a-z]+\s)?\d{4,}$"
es_numerus = df_ej["signatura_852j"].str.lower().str.fullmatch(PATRON_NUMERUS).fillna(False).astype(bool)
print("signaturas tipo número correlativo:", es_numerus.sum())
print(df_ej.loc[es_numerus, "signatura_852j"].str.extract(r"^([a-z]+)\s", expand=False)
        .value_counts(dropna=False).head(20))                 # prefijos (sur, d, …)
print(df_ej.loc[es_numerus, "localizacion_852c"].value_counts().head(20))

# Localizaciones de depósito: numeración propia por ejemplar
LOC_DEPOSITO = ["deposito", "fondo ministerio", "deposito legal"]
en_deposito = df_ej["localizacion_852c"].str.lower().isin(LOC_DEPOSITO).fillna(False).astype(bool)

print(pd.crosstab(en_deposito, es_numerus, margins=True))
# correlativos que el patrón NO detecta (revisar y ampliar PATRON_NUMERUS)
print(df_ej.loc[en_deposito & ~es_numerus, "signatura_852j"].value_counts().head(30))
# patrón detectado fuera de depósito (¿falsos positivos?)
print(df_ej.loc[~en_deposito & es_numerus, ["localizacion_852c", "signatura_852j"]].head(30))
print("vacías en depósito (no se rellenan):",
      (en_deposito & df_ej["signatura_852j"].isna()).sum())

# Prefijo de depósito sin número: signatura truncada, no válida como donante
PATRON_INCOMPLETO = r"^(?:sur|d)$"
es_incompleta = (df_ej["signatura_852j"].str.lower().str.fullmatch(PATRON_INCOMPLETO)
                   .fillna(False).astype(bool))
print("signaturas incompletas:", es_incompleta.sum())

# Criterio estricto: el depósito no dona ni recibe.
# Probado con datos: permitir donantes de depósito solo añadía 9 rellenos y 19 conflictos.
donantes_sig   = ~es_numerus & ~es_incompleta & ~en_deposito
receptores_sig = ~en_deposito

signaturas tipo número correlativo: 154205
signatura_852j
d            45539
<NA>         42863
cd           27175
sur          24378
dvd           8619
brd           2591
opo           1527
vin            775
ppe            324
rum            220
lit             54
primavera       13
otono           13
verano          11
cdr             11
invierno        10
ilo              9
laz              8
res              7
dl               7
Name: count, dtype: Int64
localizacion_852c
deposito               143017
fondo ministerio         9778
sala adultos              852
referencia                200
audiovisuales              98
periodicas                 83
deposito legal             69
sala infantil              44
fondo local. madrid        27
direccion                  12
comiteca                    8
proceso                     6
fondo manuel alvar          6
prestamo colectivo          3
exposicion                  1
lgtbi                       1
Name: count, dtype: Int64
signatura_85

In [ ]:
# 2.3 Relleno solo dentro de la misma biblioteca
# El nivel red se descarta: el 59,6 % de los títulos en ≥2 bibliotecas
# tienen signaturas distintas entre ellas.

niveles_sig = [("misma_biblioteca", ["id_registro", "sucursal_852b"])]
df, inf_sig, conf_sig = rellenar_en_cascada(df_ej, "signatura_852j", niveles_sig,
                                            donantes=donantes_sig,
                                            receptores=receptores_sig)

print(inf_sig)
print(df["signatura_852j_imputacion"].value_counts(dropna=False))

               misma_biblioteca
vacíos                    49599
rellenables                 647
rellenados                  580
en conflicto                 67
siguen vacíos             49019
signatura_852j_imputacion
<NA>                3618377
misma_biblioteca        580
Name: count, dtype: Int64


### Asignación valores nulos en subcampo

In [ ]:

m = (df["subcampo_852m"].str.lower().str.strip()
       .str.replace(r"\.$", "", regex=True)
       # prelectores: "leer. de 0 a 5 anos" es UN valor (tipo + edad), no dos
       .str.replace(r"\b(leer|jugar|aprender|mirar|idiomas)\.\s+(?=de 0 a 5 anos)",
                    r"\1 - ", regex=True))

p = m.str.split(r"\.\s+").explode().str.strip().to_frame("parte")
p["fila"] = p.index
p = p.reset_index(drop=True)
p["parte"] = p["parte"].astype("string").replace({"": pd.NA, "12-14": "de 12 a 14 anos"})

parte = p["parte"]
es_edad     = parte.str.contains(r"\banos\b|^\d+-\d+$", na=False)
es_formato  = parte.isin(["dvd", "folleto", "revista", "cd-rom"])
es_estado   = parte.isin(["duplicado", "depo", "deposito", "mostrador", "baul",
                          "taller", "depo / taller"])
es_temporal = parte.str.contains(r"\b20\d{2}\b|^novedad", na=False)
es_programa = parte.isin(["decide madrid", "speak up", "punto joven", "lgtbi", "violeta",
                          "centro interes lectura facil"])
es_genero   = parte.isin(["policiaca", "historica", "fantastica", "negra", "amor", "terror",
                          "suspense", "aventuras", "c. ficcion", "cuentos", "teatro", "series"])

p["tipo"] = pd.Series("otro", index=p.index).case_when([
    (parte.isna(), "vacío"), (es_edad, "edad"), (es_formato, "formato"),
    (es_estado, "estado"), (es_temporal, "temporal"),
    (es_programa, "programa"), (es_genero, "genero"),
])
print(p["tipo"].value_counts())
p.loc[p["tipo"].eq("otro"), "parte"].value_counts().head(50)   # ampliar listas e iterar


In [ ]:

DIMENSIONES = ["edad", "genero", "formato", "estado", "temporal", "programa", "otro"]

cols = (p[p["tipo"].ne("vacío")]
          .groupby(["fila", "tipo"])["parte"].agg(" | ".join)
          .unstack()
          .reindex(columns=DIMENSIONES)
          .reindex(df.index)
          .astype("string")
          .add_prefix("m_"))
df = df.join(cols)


In [ ]:
# ejemplares con dos valores en la misma dimensión
print(cols.apply(lambda s: s.str.contains(r" \| ", na=False).sum()))

tipo
m_edad           62
m_genero          1
m_formato         0
m_estado          0
m_temporal      304
m_programa       92
m_otro        17736
dtype: int64


In [ ]:
#Diagnóstico de las dimensiones candidatas
claves_sig_bib = ["id_registro", "sucursal_852b", "signatura_852j"]
for c in ["m_edad", "m_genero"]:
    print(c, "\n", categorias(df, c, claves_sig_bib), "\n")


m_edad 
 todos vacíos            115284
todos iguales            16357
varía                      713
mezcla valor + vacío       644
Name: count, dtype: int64 

m_genero 
 todos vacíos            129827
todos iguales             2550
mezcla valor + vacío       609
varía                       12
Name: count, dtype: int64 



In [ ]:
RELLENAR_M = ["m_edad", "m_genero"]

niveles_m = [
    ("misma_signatura",  ["id_registro", "sucursal_852b", "signatura_852j"]),
    ("misma_biblioteca", ["id_registro", "sucursal_852b"]),
]
informes_m = {}
for c in RELLENAR_M:
    df, inf, _ = rellenar_en_cascada(df, c, niveles_m)
    informes_m[c] = inf
    print(c, "\n", inf, "\n")


/tmp/ipykernel_1852/890775754.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .reindex(df.index).fillna(False).astype(bool))


m_edad 
                misma_signatura  misma_biblioteca
vacíos                 3183776           3182531
rellenables               1253              1459
rellenados                1245              1291
en conflicto                 8               168
siguen vacíos          3182531           3181240 



/tmp/ipykernel_1852/890775754.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .reindex(df.index).fillna(False).astype(bool))


m_genero 
                misma_signatura  misma_biblioteca
vacíos                 3540666           3539981
rellenables                685               722
rellenados                 685               722
en conflicto                 0                 0
siguen vacíos          3539981           3539259 



In [ ]:
# Últimas comprobaciones:
assert len(df) == len(df_or)
assert df["barcode_original"].equals(df_or["barcode_original"])
assert df["subcampo_852m"].equals(df_or["subcampo_852m"])
assert df["sucursal_852b"].equals(df_or["sucursal_852b"])

resumen = pd.DataFrame({
    c: {"vacíos antes": (df_or[c] if c in df_or else cols[c]).isna().sum(),
        "vacíos después": df[c].isna().sum(),
        "imputados": df[f"{c}_imputacion"].notna().sum()}
    for c in ["signatura_852j", *RELLENAR_M]
}).T
print(resumen)

df.info(memory_usage="deep")

                vacíos antes  vacíos después  imputados
signatura_852j         49599           49019        580
m_edad               3183776         3181240       2536
m_genero             3540666         3539259       1407
<class 'pandas.core.frame.DataFrame'>
Index: 3618957 entries, 0 to 3619005
Data columns (total 16 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   barcode_original           string
 1   id_registro                string
 2   sucursal_852b              string
 3   localizacion_852c          string
 4   signatura_852j             string
 5   subcampo_852m              string
 6   signatura_852j_imputacion  string
 7   m_edad                     string
 8   m_genero                   string
 9   m_formato                  string
 10  m_estado                   string
 11  m_temporal                 string
 12  m_programa                 string
 13  m_otro                     string
 14  m_edad_imputacion          string
 15  

### Comprobaciones:

El resultado de la asignación de los valores abiertos encontrados en subcampo muestra más campos para situar la información extraída según grupo de temática (edad, género, estado, etc...) se revisan valores y se decide si es útil para el análisis:

1. Se elimina campo signatura_852j_imputacion, hace referencia a origen imputación que ya está asignada en campo signatura.

2. Se elimina subcampo_852m, comentarios de distinta naturaleza repartidos según temática en los distintos campos.

De los cuales se eliminan los siguientes:

- campo m_temporal, muestra los comentarios sobre situaciones o actos puntuales sin referencia concreta.

- campo m_programa, muestra información general de ciclo o proyecto en el que se engloba el ejemplar, no interesa.

- campo m_estado, expresa detalles de estado del ejemplar que sin contexto no son analizables.

- m_genero_imputacion, m_edad_imputacion, informan si la imputación de los registros responde a presencia del ejemplar en la misma biblioteca y/o con la misma signatura. Se tienen en cuenta ambas.

- campo m_formato, que muestra info del formato del ejemplar específico (dvd,cd...) apuntan a formatos no libro por lo que antes se utiliza para eliminar los registros.

El resto de los campos provenientes de subcampo: m_edad (señala edades recomendadas de uso), m_genero (señala géneros literarios) y m_otros (información variada)

In [ ]:
ejemplares['m_formato'].value_counts(dropna=False)

,count
m_formato,
<NA>,3617448
revista,490
folleto,465
dvd,443
cd-rom,111


In [ ]:
# Eliminar registros con info en m_formato
print(f"Registros iniciales : {len(ejemplares)}")

valo_m_formato = ejemplares['m_formato'].notna().sum()
print(f"Registros con valor en m_formato: {valo_m_formato}")

ejemplares = ejemplares[ejemplares['m_formato'].isna()]
print(f"Registros finales: {len(ejemplares)}")

Registros iniciales : 3618957
Registros con valor en m_formato: 1509
Registros finales: 3617448


Las dimensiones iniciales del df son de (3619006, 8).

Los valores nulos no explícitos totales son 2790113 repartidos en los campos localizacion_852c (1), signatura_852j (49.599) y subcampo_852m (2.740.513)

Filas después de eliminación duplicados completos: 3.618.957

Valores nulos explícitos: 2.790.049

Filas después de eliminar ejemplares m_formato no libro: 3.617.448

In [ ]:
print(f"Columnas iniciales: {len(ejemplares.columns)}")

ejemplares = ejemplares.drop(columns=['signatura_852j_imputacion', 'subcampo_852m',
                                      'm_temporal', 'm_programa',
                                      'm_genero_imputacion', 'm_edad_imputacion',
                                      'm_estado', 'm_formato'])
print(f"Columnas finales: {len(ejemplares.columns)}")

Columnas iniciales: 16
Columnas finales: 8


### Se establecen tipos de datos

De entrada, todos son de tipo string, se mantienen como tal todos los campos menos los campos sucursal (63 valores únicos) y localizacion(40 valores únicos) que se pasa a tipo category los campos ya que son campos que presentan muy pocos valores únicos sin presentar valores nulos.

m_edad, m_genero y m_otros se mantienen string (ya pyarrow, para reducir bytes memoria), ya que no son tantos los valores únicos observados por los registros que presentan valor.

In [ ]:
# Se ven cantidad de valores únicos por columna.
ejemplares.nunique()

,0
barcode_original,3617448
id_registro,619941
sucursal_852b,63
localizacion_852c,40
signatura_852j,715750
m_edad,1402
m_genero,12
m_otro,77516


In [ ]:
ejemplares[
    ['sucursal_852b','localizacion_852c']] = ejemplares[['sucursal_852b',
                                                         'localizacion_852c']].astype("category")

### Comprobación bibliotecas en ejemplares y añadir columna id_biblioteca

Veo valores únicos en sucursal

In [ ]:
list(ejemplares['sucursal_852b'].unique())

63

In [ ]:
prprsu = pd.read_parquet(ORIGEN_LOCAL/'prprsu.parquet', engine="pyarrow")

In [ ]:
prprsu.loc[prprsu['distrito'] == 'sin_referencia', 'descripcion']

,descripcion
16,unidad central
17,bpm. biblioteca 16
18,bpm. biblioteca 12
19,bpm. biblioteca 13
24,bpm deposito general bb.pp.
36,bpm club de lectura
38,bpm despachos u. central
59,bibliotecas publicas municipales de madrid
60,bpm. biblioteca 14
61,bpm. biblioteca 15


In [ ]:

df_codigos = prprsu.drop_duplicates().merge(
    ejemplares['sucursal_852b'].drop_duplicates(),
    left_on='descripcion', right_on='sucursal_852b',
    how="outer", indicator=True
)

In [ ]:
no_coinc = df_codigos['_merge'] != 'both'
biblio = df_codigos.loc[no_coinc, ['sucursal_852b','descripcion']]
biblio.drop_duplicates().sort_values(["sucursal_852b", "descripcion"])


,sucursal_852b,descripcion
6,bpm. biblioteca 10,NaN
7,bpm. biblioteca 11,NaN
14,<NA>,bpm. biblioteca 18
18,<NA>,carabanchel. bp victoria prego
61,<NA>,villa de vallecas. bp benjamin palencia


Discrepancia entre las bibliotecas que hay en las tablas prprsu y ejemplares.

3 de las bibliotecas referenciadas en prprsu no aparecen en la tabla ejemplares, no constan ejemplares para ellas.

- bpm. biblioteca 18
- carabanchel. bp victoria prego
- villa de vallecas. bp benjamin palencia

2 de las bibliotecas nombradas en ejemplares no aparecen entre las referencias de la tabla prprsu.

Me guardo lista de ejemplares asignados a estas para comparar con prestamos:
- bpm. biblioteca 10 -> tiene 37 ejemplares asignados.
- bpm. biblioteca 11 -> tiene 37 ejemplares asignados.

In [ ]:
# Lista de ejemplares en la biblioteca 'bpm. biblioteca 10'

# máscara
b_10 = ejemplares['sucursal_852b'] == 'bpm. biblioteca 10'

# id_ejemplares únicos en b_10

ejemplares_b_10 = ejemplares.loc[b_10, 'barcode_original']
lista_b_10 = list(ejemplares_b_10)
len(lista_b_10)

37

In [ ]:
# Lista de ejemplares en la biblioteca 'bpm. biblioteca 11'

# máscara
b_11 = ejemplares['sucursal_852b'] == 'bpm. biblioteca 11'

# id_ejemplares únicos en b_11

ejemplares_b_11 = ejemplares.loc[b_11, 'barcode_original']
lista_b_11 = list(ejemplares_b_11)
len(lista_b_11)

37

### Se añade id_biblioteca a la tabla ejemplares

In [ ]:

ejemplares = ejemplares.merge(
    prprsu[["id_biblioteca", "descripcion"]],
    left_on="sucursal_852b", right_on="descripcion",
    how="left",
    validate="m:1",
    indicator=True,
)

display(ejemplares["_merge"].value_counts())
print(ejemplares.loc[ejemplares["_merge"] == "left_only", "sucursal_852b"].unique())

,count
_merge,
both,3617374
left_only,74
right_only,0


<ArrowStringArray>
['bpm. biblioteca 10', 'bpm. biblioteca 11']
Length: 2, dtype: string


In [ ]:
# Comprobación
display(ejemplares["_merge"].value_counts())
print(ejemplares.loc[ejemplares["_merge"] == "left_only", "sucursal_852b"].unique())
print(ejemplares.shape)

,count
_merge,
both,3617374
left_only,74
right_only,0


<ArrowStringArray>
['bpm. biblioteca 10', 'bpm. biblioteca 11']
Length: 2, dtype: string
(3617448, 11)


In [ ]:
ejemplares.isna().sum()

,0
barcode_original,0
id_registro,0
sucursal_852b,0
localizacion_852c,0
signatura_852j,49019
m_edad,3179731
m_genero,3537750
m_otro,3297883
id_biblioteca,74
descripcion,74


## Eliminación de columnas y cambios nombre

In [ ]:
columnas = {
    'barcode_original':'id_ejemplar',
    'id_registro':'id_registro',
    'sucursal_852b' :'biblioteca',
    'id_biblioteca':'id_biblioteca',
    'localizacion_852c':'loc_ejemplar',
    'signatura_852j':'tejuelo',
    'm_edad':'info_edad',
    'm_genero':'info_genero',
    'm_otro':'info_otro'
}

ejemplares = ejemplares[list(columnas)].rename(columns=columnas, errors="raise")

In [ ]:
# Se vuelve a poner como category al campo biblioteca
ejemplares['biblioteca'] = ejemplares['biblioteca'].astype("category")

In [ ]:
print(ejemplares.shape, ejemplares.info())
display(ejemplares.isna().sum(), ejemplares.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3617448 entries, 0 to 3617447
Data columns (total 9 columns):
 #   Column         Dtype   
---  ------         -----   
 0   id_ejemplar    string  
 1   id_registro    string  
 2   biblioteca     category
 3   id_biblioteca  string  
 4   loc_ejemplar   category
 5   tejuelo        string  
 6   info_edad      string  
 7   info_genero    string  
 8   info_otro      string  
dtypes: category(2), string(7)
memory usage: 361.6 MB
(3617448, 9) None


,0
id_ejemplar,0
id_registro,0
biblioteca,0
id_biblioteca,74
loc_ejemplar,0
tejuelo,49019
info_edad,3179731
info_genero,3537750
info_otro,3297883


,0
id_ejemplar,3617448
id_registro,619941
biblioteca,63
id_biblioteca,61
loc_ejemplar,40
tejuelo,715750
info_edad,1402
info_genero,12
info_otro,77516


Tabla final:

3.617.448 ejemplares que engloban 619.941 registros bibliotecarios, repartidos en 63 bibliotecas (61 de las que hay info en prprsu).

## Exportación parquet

In [ ]:
ejemplares.to_parquet(
    DRIVE_DESTINO / 'ejemplares_2.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

# Registros - Limpiezas particulares



In [ ]:
# Se determina nombre de búsqueda df:
nombre = "registros_v4"
# Se extrae df ejemplares del diccionario del catálogo:
df_reg = cat_dfs[nombre]

In [ ]:
# Valores iniciales antes de las transformaciones:
print(f"Las dimensiones iniciales del df son de {df_reg.shape}."
#f"\nLos valores nulos totales son: {cat_resumen_nulos[nombre].loc['total', 'total']}"
"\nRepartidos como se muestra en la siguiente tabla:\n")


Las dimensiones inciales del df son de (656404, 41).
Repartidos como se muestra en la siguiente tabla:



### Eliminación registros no libros


Antes de nada, se filtrará por los registros que pertenecen a los de tipo material textual monográfico, dentro de los cuales se contemplan los libros. Estos registros están referenciados por los campos 6 y 7 presentes en la cabecera de los registros introducidos en el catálogo.

Filtrando por el campo es_libro, por aquellos valores = 1. Se eliminan los registros con valor 0, los cuales no son material textual monográfico.

In [ ]:
# Se miran valores únicos del campo es_libro
# No se selecciona que tenga en cuenta nulos porque aun no se han establecido
# los valores nulos explícitos. Si los hubiera (dropna=False)
print(f"En el campo es_libros hay {df_reg['es_libro'].nunique()} valores. "
f"Siendo estos: {list(df_reg['es_libro'].unique())} ")

# Se ven cantidad de registros con valor 0 (no libro)
# Se observa suma de valores con value_counts y se filtra por etiqueta
# Como el dtype del campo es string se ha de señalar el 0 como tal
no_libro = df_reg['es_libro'].value_counts().loc['0',]

# Vemos registros que desapareceran
print(f"De los {len(df_reg)} registros, {no_libro} NO son Material Textual")



En el campo es_libros hay 2 valores. Siendo estos: ['1', '0'] 
De los 656404 registros, 141030 NO son Material Textual


In [ ]:
# Se procede a filtrar:

# Máscara de los registros que si son libros:
reg_libros = df_reg['es_libro'] == "1"
# máscara sobre df, siempre mismo len que df_ref. Muestra true-false dependiendo
# de si cumple condición

# Comprobación:
# solo se tienen en cuenta los true de la máscara
assert reg_libros.sum() + no_libro == len(df_reg)

# Establecer el df con máscara:
registros = df_reg.loc[reg_libros].copy()
print(f"Registros totales post filtrado: {len(registros)}")

del df_reg


Registros totales post filtrado: 515374


### Conversión valores nulos explícitos

Para ver qué % de valores no nulos tiene cada campo paso a nulo explícito los valores contemplados en POSIBLES_NULOS, excluyendo "x", "xx" y "xxx" que se revisarán en su momento.

In [ ]:
# Resumen nulos

posibles_nulos = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan","sd", "s/d", "sin datos",
                  "tbd")

nulos_posibles = buscar_nulos(registros, posibles_nulos)
display(nulos_posibles)


columna,anio_publicacion,autor,cdu_080,critica_literaria_cdu,fecha1_008,fecha2_008,forma_literaria_008,forma_literaria_cdu,genero_literario_cdu,generos_655,...,subtitulo_245b,tipo_fecha_008,tipo_publico_521,titulo_245a,titulo_orden_245a,titulo_original_nota,titulo_uniforme_130,titulo_uniforme_240,traduccion_041_ind1,total
'',31250,78646,33632,510024,342,891,1478,303924,303924,368931,...,357843,334,463674,64,96,514021,513488,495468,465211,8913795
'-',0,0,0,0,0,0,0,0,0,0,...,0,4,0,0,0,0,0,0,0,7
'.',0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,2,0,0,0,0,3
'?',0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
'na',0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
total,31250,78646,33632,510024,342,891,1479,303924,303924,368931,...,357843,339,463674,64,99,514021,513488,495468,465211,8913807


In [ ]:
print(f"Total posibles nulos {nulos_posibles.loc["total","total"]}")


registros = registros.replace(posibles_nulos, pd.NA)

print(f"Total nulos convertidos: {registros.isna().sum().sum()}")
del nulos_posibles

Total posibles nulos 8913807
Total nulos convertidos: 8913807


### Eliminación de los campos no útiles

In [ ]:
# Valoración datos nulos, cantidad y porcentaje según registros:

resumen_campo = pd.DataFrame({
    'no_nulos': registros.notna().sum(),
    'pct_no_nulos': (registros.notna().mean() * 100).round(2),
    'dtype': registros.dtypes
}).sort_values('pct_no_nulos')
display(resumen_campo)
del resumen_campo

,no_nulos,pct_no_nulos,dtype
lengua_intermedia_041k,141,0.03,string[python]
titulo_original_nota,1353,0.26,string[python]
titulo_uniforme_130,1886,0.37,string[python]
critica_literaria_cdu,5350,1.04,string[python]
lengua_240l,14892,2.89,string[python]
parte_245np,16561,3.21,string[python]
titulo_uniforme_240,19906,3.86,string[python]
lengua_original_041h,45450,8.82,string[python]
traduccion_041_ind1,50163,9.73,string[python]
lengua_texto_041a,50136,9.73,string[python]


In [ ]:
campos_eliminar = ['ld06', 'ld07', 'subtitulo_245b','lugar_literatura_cdu',
                   'nota_publico_521', 'fecha2_008', 'titulo_orden_245a',
                   'lengua_intermedia_041k', 'titulo_original_nota', 'titulo_uniforme_130',
                   'titulo_uniforme_240','critica_literaria_cdu', 'lengua_240l', 'parte_245np',
                   'lengua_original_041h', 'lengua_texto_041a','lengua_literatura_cdu',
                   'tipo_publico_521', 'publico_destinatario_521a',
                   'es_libro', 'traduccion_041_ind1']
sorted(campos_eliminar)

['critica_literaria_cdu',
 'es_libro',
 'fecha2_008',
 'ld06',
 'ld07',
 'lengua_240l',
 'lengua_intermedia_041k',
 'lengua_literatura_cdu',
 'lengua_original_041h',
 'lengua_texto_041a',
 'lugar_literatura_cdu',
 'nota_publico_521',
 'parte_245np',
 'publico_destinatario_521a',
 'subtitulo_245b',
 'tipo_publico_521',
 'titulo_orden_245a',
 'titulo_original_nota',
 'titulo_uniforme_130',
 'titulo_uniforme_240',
 'traduccion_041_ind1']

In [ ]:
# Se comprueba que la lista de campos a eliminar contenga solo valores únicos.
# Y que después de comprobarlo tenga la misma cantidad de valores.
assert len(campos_eliminar) == len(set(campos_eliminar))
print(f"Columnas iniciales: {registros.shape[1]}."
f"\nColumnas a eliminar: {len(campos_eliminar)}")
registros = registros.drop(columns=campos_eliminar, errors="ignore")
print(f"Columnas finales : {len(registros.columns)}")

del campos_eliminar

Columnas iniciales: 41.
Columnas a eliminar: 21
Columnas finales : 20


### Duplicados completos

In [ ]:
print(f"Registros totales: {len(registros)}")
# todos los registros duplicados:
duplis_completos = registros[registros.duplicated(keep=False)]

# Se mira cantidad de registros completamente duplicados:
print(f"Hay {len(duplis_completos)} registros totalmente duplicados")

# Se mira cantidad de grupos de registros completamente duplicados:
tam = duplis_completos.value_counts(subset='id_registro',dropna=False) # Duplicados teniendo en cuenta nulos
n_grupos = len(tam)
print(f"Numero de grupos registros duplicados: {len(tam)}"
"\nLos grupos se distribuyen de la siguiente manera: \n")
display(tam.value_counts())
reg_eliminar = tam.sum() - len(tam)
print(f"\nSe eliminaran {reg_eliminar} registros."
f"Dejando {len(registros) - reg_eliminar} registros restantes")

Registros totales: 515374
Hay 2 registros totalmente duplicados
Numero de grupos registros duplicados: 1
Los grupos se distribuyen de la siguiente manera: 



,count
count,
2,1



Se eliminaran 1 registros.Dejando 515373 registros restantes


In [ ]:
#Eliminación registro:
registros = registros.drop_duplicates()

print(f"Registros finales: {registros.shape[0]}")

Registros finales: 515373


In [ ]:
del duplis_completos, tam

### Duplicados por clave - id_registro

In [ ]:
# Duplicados en código de barras:
clave  = 'id_registro'
duplis_clave = registros[registros.duplicated(subset=clave, keep=False)]
n_duplis_calve = len(duplis_clave)
if n_duplis_calve == 0:
  print("No hay registros duplicados según la clave")
else:
  tam2 = duplis_clave.value_counts(subset=clave, dropna=False)
  print(f"Se han encontrado los {n_duplis_calve}"
        f" con la clave - {clave} duplicada")
  display(tam2)

Se han encontrado los 2 con la clave - id_registro duplicada


,count
id_registro,
es-marbcm02153130c-0,2


In [ ]:
# Visualizar registros
registros.loc[registros['id_registro'] == 'es-marbcm02153130c-0']

,id_registro,len_008,tipo_fecha_008,fecha1_008,pais_edicion_008,lengua_texto_008,publico_008,forma_literaria_008,traductor_700,mencion_traduccion_245c,titulo_245a,autor,isbn_original,anio_publicacion,cdu_080,forma_literaria_cdu,genero_literario_cdu,materias_650,generos_655,n_ejemplares
642290,es-marbcm02153130c-0,38,e,2024,sp,spa,e,0,0,0,la magia de la ruinas,"zuchtriegel, gabriel",9788430626991,2024,904(377):7.032(37)|39(377),<NA>,<NA>,<NA>,<NA>,44
642507,es-marbcm02153130c-0,38,e,2024,sp,spa,e,0,0,0,la magia de las ruinas,"zuchtriegel, gabriel",9788430626991,2024,904(377):7.032(37)|39(377),<NA>,<NA>,<NA>,<NA>,44


In [ ]:
# Se buscan diferencias:
a = registros.loc[642290]   # primer registro (por etiqueta de índice)
b = registros.loc[642507]   # segundo registro

a.compare(b)

,self,other
titulo_245a,la magia de la ruinas,la magia de las ruinas


Se han encontrado dos registros con mismo id_registro. Se han buscado así como sus diferencias. Se observa que en el primer registro índice 642290 hay una errata en el título, falta una s las (las ruinas). Se elimina. Pendiente de revisar en tabla ejemplares + prestamos si aparece título erróneo.

In [ ]:
# Se localiza registro por título erróneo:
tit_incorrecto = registros['titulo_245a'] == 'la magia de la ruinas'

# Se buscan más registros con ese título:
display(registros.loc[tit_incorrecto])
print(f"Registros iniciales {registros.shape[0]}.")
print(f"Hay {len(registros.loc[registros['id_registro'] == 'es-marbcm02153130c-0'])}"
f" registros con {clave} duplicado."
f"Hay {len(registros.loc[tit_incorrecto])} registros con titulo incorrecto")
# Solo hay el contemplado

# Arreglo del título y eliminación de registro duplicado
registros.loc[registros["titulo_245a"] == "la magia de la ruinas", "titulo_245a"] = "la magia de las ruinas"
registros = registros.drop_duplicates()
print(f"Registros finales: {len(registros)}")

,id_registro,len_008,tipo_fecha_008,fecha1_008,pais_edicion_008,lengua_texto_008,publico_008,forma_literaria_008,traductor_700,mencion_traduccion_245c,titulo_245a,autor,isbn_original,anio_publicacion,cdu_080,forma_literaria_cdu,genero_literario_cdu,materias_650,generos_655,n_ejemplares
642290,es-marbcm02153130c-0,38,e,2024,sp,spa,e,0,0,0,la magia de la ruinas,"zuchtriegel, gabriel",9788430626991,2024,904(377):7.032(37)|39(377),<NA>,<NA>,<NA>,<NA>,44


Registros iniciales 515373.
Hay 2 registros con id_registro duplicado.Hay 1 registros con titulo incorrecto
Registros finales: 515372


Las dimensiones iniciales del df son de 656.404 registros


Dimensiones df filtrado por libros 515.374 registros

Dimensiones df después eliminación duplicados completos 515.373 registros.


Dimensiones df después eliminación id_registro duplicado  515.372 registros.

### Unificación tablas dimensiones con campos de Registros

Se proceden a unificar las siguientes tablas soporte con sus respectivos campos de la tabla registros:

Durante el proceso de transformación y unión

1. df_dc1 : cod_cdu_forma_literaria -> forma_literaria_cdu
2. df_dc2 : cod_forma_literaria_008 -> forma_literaria_008
3. df_dc3 : cod_lengua_marc -> lengua_texto_008
4. df_dc4: cod_pais_marc -> pais_edicion_008
5. df_dc5 : cod_publico_008 -> publico_008
6. df_dc6 : cod_tipo_fecha_008 -> tipo_fecha_008

En la etapa bronze había las siguientes tablas soporte para los campos eliminados:

- cod_cdu_lengua_literatura -> lengua_literatura_cdu -> Tiene muchos errores en la codificación de los valores CDU. Con cobertura de poco menos de la mitad de los registros se toma como referencia útil el código 008.
- cod_cdu_lugar_literatura -> lugar_literatura_cdu
- cod_tipo_publico_521_ind1 -> tipo_publico_521
- cod_traduccion_041_ind1 -> traduccion_041_ind1

Todos los campos de las tablas de dimensión que se unirán a esta son de tipo string. Así mismo, todos los códigos y descripciones son valores únicos y no presentan valores nulos.

Se pasan los campos presentes en esta tabla a formato category antes de la asignación. Los nuevos campos también se pasarán a category una vez asignados todos los valores.

### **1. df_dc1 : cod_cdu_forma_literaria -> forma_literaria_cdu**

In [ ]:
# Se establece variable para ir sustituyendo
soporte = df_dc1 #DF tabla soporte
campo = "forma_literaria_cdu"
nuevo_campo = campo + "_descrito"
# Se comprueban valores únicos en código de tabla soporte
print(list(soporte['codigo'].unique()))

# Y nombre campos de tabla soporte
print(list(soporte.columns))

['1', '2', '3', '31', '32', '34', '4', '5', '6', '7', '8', '82', '83', '85', '9', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '0']
['codigo', 'descripcion', 'verificado']


In [ ]:
# Cambio tipo de dato a campo
registros[campo] = registros[campo].astype("category")

In [ ]:
# Nulos en campo
print(f"Antes de la asignación hay {registros[campo].isna().sum()} valores nulos")

Antes de la asginación hay 303923 valores nulos


Se observa que 303.923 registros con valor nulo código establecido para este campo, no constando la información en el catálogo.

Más adelante se mirará si es asignable de alguna manera.

In [ ]:
# Se comprueba que valores en campo de registros no están en los códigos de la tabla:

# Se establece filtro invertido -> se busca todos los registros que NO cumplen condición
sin_descripcion = ~registros[campo].isin(soporte["codigo"]) & ~registros[campo].isna()

# Se contempla Serie tratada en registros para los valores reales no encontrados
# en tabla soporte:
codigos_no_coincidente = registros.loc[sin_descripcion, campo]
print(f"Hay los siguientes registros según valor no coincidente: \n{codigos_no_coincidente.value_counts().sum()}"
" \n Cuyos valores son  distintos a los contemplados en tabla soporte.\n"
f"Los valores no coincidentes son: {codigos_no_coincidente.unique()}")


Hay los siguientes registros según valor no coincidente: 
1985 
 Cuyos valores son  distintos a los contemplados en tabla soporte.
Los valores no coincidentes son: ['1|3', '6|83', '3|82', '3|99', '1|2', ..., '1|3|82', '91|99', '9|91', '96|4', '7|92']
Length: 167
Categories (190, string): [0, 0|3, 0|4, 1, ..., 99|6, 99|94, 9|91, 9|99]


Se observa que hay 1.985 valores presentes en el campo observado que no se encuentran en la tabla descriptiva.

Son valores con códigos concatenados y valores únicos no contemplados en tabla soporte.

Se analiza composición.

In [ ]:
# Celdas con más de un valor (separador "|")
multi_descripcion = registros[campo].str.contains("|", regex=False,  na=False)
print(f"{campo}: {multi_descripcion.sum()} celdas con varios valores")

# Celdas con un único valor, no coincidente con tabla soporte:

# Se añade filtro invertido del multi_descripcion al de códigos descritos
codigos_unicos_desconocidos = sin_descripcion & ~multi_descripcion
codigos_desconocidos = registros.loc[codigos_unicos_desconocidos, campo]
print(f"Por último, se han encontrado {len(codigos_desconocidos)} registros, "
"con valores de códigos no presentes en tabla soporte.")

lista_codigos_desconocidos = codigos_desconocidos.unique()

forma_literaria_cdu: 1241 celdas con varios valores
Por último, se han encontrado 744 registros, con valores de códigos no presentes en tabla soporte.


In [ ]:
# Sustitución de los códigos por la descripción:

# se establece el código como índice para hacer el cruce a partir de él y queda como
# único campo la descripción. Como diccionario
mapa = soporte.set_index("codigo")["descripcion"]

registros[nuevo_campo] = registros[campo].map(mapa)



In [ ]:
# Revisión

# Se ven valores únicos, teniendo en cuenta nulos, y cantidades
valores_unicos_nuevo_campo = registros[nuevo_campo].value_counts(dropna=False)

# Se ve cantidad de nulos en campo y no nulos:
valores_nulos_nuevo_campo = registros[nuevo_campo].isna().sum()
valores_nuevo_campo = registros[nuevo_campo].value_counts().sum()

# Se confirma que total registros compuestos por nulos y no nulos
assert len(registros) == valores_nulos_nuevo_campo + valores_nuevo_campo

# Se ven origen de todos los nulos en nuevo campo:
assert valores_nulos_nuevo_campo == multi_descripcion.sum() + len(codigos_desconocidos) + registros[campo].isna().sum()

print(f"Hay {valores_nuevo_campo} registros validos en {nuevo_campo}.\n")
print(f"Han aparecido {valores_nulos_nuevo_campo} valores nulos en {nuevo_campo}.\n"
f"\n - {registros[campo].isna().sum()} vienen de valores nulos en {campo}."
f"\n - {valores_nulos_nuevo_campo - registros[campo].isna().sum()} valores "
f" no corresponden a valores nulos en {campo}:"
f"\n   - {multi_descripcion.sum()} corresponden a registros con varios códigos por celda."
f"\n   - {len(codigos_desconocidos)} corresponden a códigos desconocidos que no aparecen en la tabla soporte")

Hay 209464 registros validos en forma_literaria_cdu_descrito.

Han aparecido 305908 valores nulos en forma_literaria_cdu_descrito.

 - 303923 vienen de valores nulos en forma_literaria_cdu.
 - 1985 valores  no corresponden a valores nulos en forma_literaria_cdu:
   - 1241 corresponden a registros con varios códigos por celda.
   - 744 corresponden a códigos desconocidos que no aparecen en la tabla soporte


Hay 209464 registros válidos en forma_literaria_cdu_descrito.

Han aparecido 305909 valores nulos en forma_literaria_cdu_descrito.

 - 303924 vienen de valores nulos en forma_literaria_cdu.
 - 1985 valores  no corresponden a valores nulos en forma_literaria_cdu:
   - 1241 corresponden a registros con varios códigos por celda. -> Se procede a realizar transcripción en campo descripción, tomando solo el primer código.
   - 744 corresponden a códigos desconocidos que no aparecen en la tabla soporte. ->  Debido a una incorrecta interpretación del código CDU por falta de información en todos sus parámetros posibles. Al suponer un porcentaje casi 0 de registros se determinan como no conocido.
    





In [ ]:
# Se ven valores de los códigos desconocidos:
print(codigos_desconocidos.unique())

# Es un único código -> 84
codigo_unico = registros['forma_literaria_cdu'] == '84'

# Asignación valor:
registros.loc[codigo_unico, nuevo_campo] = 'codigo desconocido'

print(f"Valores nulos en {nuevo_campo}: {registros[nuevo_campo].isna().sum()}"
f"\nHan disminuido {valores_nulos_nuevo_campo - registros[nuevo_campo].isna().sum()}")

['84']
Categories (190, string): [0, 0|3, 0|4, 1, ..., 99|6, 99|94, 9|91, 9|99]
Valores nulos en forma_literaria_cdu_descrito: 305164
Han disminuido 744


Se asigna descripción a los registros con más de uno asignado
Tomando el primer válido.

In [ ]:
# Asignación descripción multivalor:
print(f"Valores nulos en {nuevo_campo} antes de la asignación: "
f" {registros[nuevo_campo].isna().sum()}")

EXCLUIR = {}   # códigos que no deben elegirse como forma principal (artefacto)

dic = mapa.to_dict()   # el diccionario es más rápido y seguro que buscar en la Serie

def primera_desc_valida(celda, sep="|"):
    """Devuelve la descripción del primer código de la celda que está en la tabla y no está excluido."""
    for c in celda.split(sep):
        c = c.strip()
        if c in dic and c not in EXCLUIR:
            return dic[c]
    return pd.NA   # ningún código válido: queda como nulo para el control

# Máscara de filas multivalor (na=False por seguridad si la columna tiene algún NaN)
multi_descripcion = registros[campo].str.contains("|", regex=False, na=False)

# Solo se rellenan las filas multivalor; el resto conserva lo asignado con .map(mapa)
registros.loc[multi_descripcion, nuevo_campo] = (
    registros.loc[multi_descripcion, campo].map(primera_desc_valida)
)

# --- Control ---
print(f"Filas multivalor: {multi_descripcion.sum():,}")
print(f"Multivalor sin código válido: {registros.loc[multi_descripcion, nuevo_campo].isna().sum():,}")
print(f"Valores nulos finales en {nuevo_campo}: {registros[nuevo_campo].isna().sum()}")
print(registros.loc[multi_descripcion & registros[nuevo_campo].isna(), campo].value_counts().head())

Valores nulos en forma_literaria_cdu_descrito antes de la asignación:  305164
Filas multivalor: 1,241
Multivalor sin código válido: 0
Valores nulos finales en forma_literaria_cdu_descrito: 303923
forma_literaria_cdu
0      0
0|3    0
0|4    0
1      0
1|2    0
Name: count, dtype: int64


In [ ]:
# Cambio tipo de dato nuevo campo a category:
registros[nuevo_campo] = registros[nuevo_campo].astype("category")


valores_unicos_nuevo_campo

,count
forma_literaria_cdu_descrito,
<NA>,305908
prosa narrativa,149017
poesia,21926
comic,18372
teatro,7570
"memorias, biografia, historiografia",3313
otras formas literarias (82-9),2276
ensayo,2245
antologias,1011


El campo forma_literaria_cdu comparte funcionalidad con el campo forma_literaria_008. Se asignará valor una vez contemplada la unificación de los códigos de ese campo con su tabla y analizar si más útil campo forma_literaria_008.

### **2. df_dc2 : cod_forma_literaria_008 -> forma_literaria_008**

In [ ]:
# Se establece variable para ir sustituyendo

# DF tabla soporte
soporte = df_dc2

# Nombre campo

campo = "forma_literaria_008"
nuevo_campo = campo + "_descrito"

# Se comprueban valores únicos en código de tabla soporte
print(list(soporte['codigo'].unique()))

# Y nombre campos de tabla soporte
print(list(soporte.columns))

# Cambio tipo de dato a campo
registros[campo] = registros[campo].astype("category")

# Nulos en campo
print(f"Antes de la asignación hay {registros[campo].isna().sum()} valores nulos")

['0', '1', 'c', 'd', 'e', 'f', 'h', 'i', 'j', 'm', 'p', 's', 'u', '|']
['codigo', 'descripcion', 'fuente']
Antes de la asginación hay 1479 valores nulos


Se observa que 1.479 valores nulos. No constando información en el catálogo.

Más adelante se mirará si es asignable de alguna manera.

In [ ]:
# Se comprueba que valores en campo de registros no están en los códigos de la tabla:

# Se establece filtro invertido -> se busca todos los registros que NO cumplen condición
sin_descripcion = ~registros[campo].isin(soporte["codigo"]) & ~registros[campo].isna()

# Se contempla Serie tratada en registros para los valores reales no encontrados
# en tabla soporte:
codigos_no_coincidente = registros.loc[sin_descripcion, campo]
print(f"Hay los siguientes registros según valor no coincidente: \n{codigos_no_coincidente.value_counts().sum()}"
" \n Cuyos valores son  distintos a los contemplados en tabla soporte.\n"
f"Los valores no coincidentes son: {codigos_no_coincidente.unique()}")


Hay los siguientes registros según valor no coincidente: 
56264 
 Cuyos valores son  distintos a los contemplados en tabla soporte.
Los valores no coincidentes son: ['#', 'l', 'b', 'a', 'v', ..., '9', '4', 'º', 't', 'q']
Length: 19
Categories (33, string): [!, #, ', +, ..., u, v, |, º]


In [ ]:
# Se ve cantidad de valores #

valor_des_ni = codigos_no_coincidente.value_counts().loc["#"]
print(f"Hay {valor_des_ni} correspondientes a 'blanco - no valido (#)'")
print(f"Dejando {codigos_no_coincidente.value_counts().sum() - valor_des_ni}"
" pendientes de determinar.")


Hay 55795 correspondientes a 'blanco - no valido (#)'
Dejando 469 pendientes de determinar.


Se observa que hay 56.264 valores presentes en el campo observado que no se encuentran en la tabla soporte.

Entre estos valores no presentes en tabla soporte, 55.795 presentan valor #, según  uso del catálogo, "#" corresponde a desconocido no especificado. Se añade el valor de blanco(no válido) al map que se realiza para asignar descripciones.

El resto de valores, 469, corresponden a incorrecta codificación se asignan como tal.

Se analiza composición.

In [ ]:
# Sustitución de los códigos por la descripción:

# Se establece el código como índice para hacer el cruce a partir de él y queda como
# único campo la descripción. Como diccionario
mapa = soporte.set_index("codigo")["descripcion"]
mapa.loc["#"] = "blanco(no valido)"

registros[nuevo_campo] = registros[campo].map(mapa)

# Se añade comentario para los mal codificados:
blanco = sin_descripcion & ~registros[campo].isin(["#"])
registros.loc[blanco, nuevo_campo] = "codigo_corrupto"

# Cambio tipo de dato nuevo campo a category:
registros[nuevo_campo] = registros[nuevo_campo].astype("category")


In [ ]:
# Se revisa cantidad de nulos después de la primera asignación :

# Se ve cantidad de nulos en campo y no nulos:
valores_nulos_nuevo_campo = registros[nuevo_campo].isna().sum()
valores_nuevo_campo = registros[nuevo_campo].value_counts().sum()
etiquetas_nuevo_campo = registros[nuevo_campo].unique()

# Revisiones:
assert valores_nulos_nuevo_campo == registros[campo].isna().sum()

assert len(registros) == valores_nulos_nuevo_campo + valores_nuevo_campo

total_grupo = registros[nuevo_campo].value_counts(dropna=False).reset_index()
total_grupo["%"] = ((total_grupo["count"] / len(registros))*100).round(2)
total_grupo = total_grupo.set_index(nuevo_campo)
print(f"Se han encontrado {valores_nulos_nuevo_campo} valores nulos en la conversión."
f" Que corresponde a un {total_grupo.iloc[7,1]} % de los registros.\n"
f"\nCon total de {valores_nuevo_campo} registros correctamente convertidos."
f"\nCompuestos de las siguientes etiquetas: \n{list(etiquetas_nuevo_campo)}"
f"\n\nSin embargo, en un total del "
f" {total_grupo.loc[["blanco(no valido)",'no codificado'], '%'].sum().round(2)} "
"% de los registros falta descripión por falta de codificación o por ser desconocido.")


Se han encontrado 1479 valores nulos en la conversión. Que corresponde a un 0.29 % de los registros.

Con total de 513893 registros correctamente convertidos.
Compuestos de las siguientes etiquetas: 
['no ficcion', 'ficcion (sin especificar)', 'blanco(no valido)', 'novela', 'no codificado', 'codigo_corrupto', 'teatro', nan, 'poesia', 'relatos cortos', 'desconocida', 'ensayo', 'humor, satira', 'formas mixtas', 'comics / historietas', 'discursos', 'cartas']

Sin embargo, en un total del  30.12 % de los registros falta descripión por falta de codificación o por ser desconocido.


In [ ]:
display(total_grupo)

,count,%
forma_literaria_008_descrito,,
no ficcion,196266,38.08
ficcion (sin especificar),122505,23.77
no codificado,99416,19.29
blanco(no valido),55795,10.83
novela,28977,5.62
relatos cortos,3690,0.72
poesia,3665,0.71
NaN,1479,0.29
teatro,1347,0.26


### **3. df_dc3 : cod_lengua_marc -> lengua_texto_008**

In [ ]:
# Se establece variable para ir sustituyendo
soporte = df_dc3 #DF tabla soporte
campo = "lengua_texto_008"
nuevo_campo = campo + "_descrito"

# Se comprueban valores únicos en código de tabla soporte
print(list(soporte['codigo'].unique()))

# Y nombre campos de tabla soporte
print(list(soporte.columns))

['abk', 'ace', 'ach', 'ada', 'ady', 'aar', 'afh', 'afr', 'afa', 'ain', 'aka', 'akk', 'alb', 'ale', 'alg', 'ajm', 'alt', 'tut', 'amh', 'anp', 'apa', 'ara', 'arg', 'arc', 'arp', 'arw', 'arm', 'rup', 'art', 'asm', 'ath', 'aus', 'map', 'ava', 'ave', 'awa', 'aym', 'aze', 'ast', 'ban', 'bat', 'bal', 'bam', 'bai', 'bad', 'bnt', 'bas', 'bak', 'baq', 'btk', 'bej', 'bel', 'bem', 'ben', 'ber', 'bho', 'bih', 'bik', 'byn', 'bis', 'zbl', 'bos', 'bra', 'bre', 'bug', 'bul', 'bua', 'bur', 'cad', 'car', 'cat', 'cau', 'ceb', 'cel', 'cai', 'chg', 'cmc', 'cha', 'che', 'chr', 'chy', 'chb', 'chi', 'chn', 'chp', 'cho', 'chu', 'chk', 'chv', 'cop', 'cor', 'cos', 'cre', 'mus', 'crp', 'cpe', 'cpf', 'cpp', 'crh', 'hrv', 'cus', 'cze', 'dak', 'dan', 'dar', 'day', 'del', 'din', 'div', 'doi', 'dra', 'dua', 'dut', 'dum', 'dyu', 'dzo', 'frs', 'bin', 'efi', 'egy', 'eka', 'elx', 'eng', 'enm', 'ang', 'myv', 'esk', 'epo', 'est', 'gez', 'ewe', 'ewo', 'fan', 'fat', 'fao', 'fij', 'fil', 'fin', 'fiu', 'fon', 'fre', 'frm', 'fro'

In [ ]:
# Cambio tipo de dato a campo
registros[campo] = registros[campo].astype("category")

# Nulos en campo
print(f"Antes de la asignación hay {registros[campo].isna().sum()} valores nulos")

Antes de la asginación hay 1629 valores nulos


Se observa que 1.629 registros con valor nulo en código establecido para este campo, no constando la información en el catálogo.

Más adelante se mirará si es asignable de alguna manera.

In [ ]:
# Se comprueba que valores en campo de registros no están en los códigos de la tabla:

# Se establece filtro invertido -> se busca todos los registros que NO cumplen condición
sin_descripcion = ~registros[campo].isin(soporte["codigo"]) & ~registros[campo].isna()

# Se contempla Serie tratada en registros para los valores reales no encontrados
# en tabla soporte:
codigos_no_coincidente = registros.loc[sin_descripcion, campo]
print(f"Hay los siguientes registros según valor no coincidente: \n{codigos_no_coincidente.value_counts().sum()}"
" \n Cuyos valores son  distintos a los contemplados en tabla soporte.\n"
f"Los valores no coincidentes son: {codigos_no_coincidente.unique()}")


Hay los siguientes registros según valor no coincidente: 
597 
 Cuyos valores son  distintos a los contemplados en tabla soporte.
Los valores no coincidentes son: ['no#', '###', 'pa#', 'val', 'spc', ..., 'ng#', 'esp', 'gal', 'deu', '///']
Length: 60
Categories (123, string): [###, #es, #sp, ///, ..., zxx, zzz, |##, |||]


Se observa que hay 597 valores presentes en el campo observado que no se encuentran en la tabla descriptiva.

Representan un % ínfimo en la totalidad de registros. Se dejan como nulos por incorrecta codificación del catálogo, señalándose así en el nuevo campo.



In [ ]:
# Sustitución de los códigos por la descripción:

# se establece el código como índice para hacer el cruce a partir de él y queda como
# único campo la descripción. Como diccionario
mapa = soporte.set_index("codigo")["nombre_en"]

registros[nuevo_campo] = registros[campo].map(mapa)

# Se añade comentario para los mal codificados:
registros.loc[sin_descripcion, nuevo_campo] = "codigo_corrupto"

# Cambio tipo de dato nuevo campo a category:
registros[nuevo_campo] = registros[nuevo_campo].astype("category")


In [ ]:
valores_unicos_nuevo_campo

,count
lengua_texto_008_descrito,
spanish,465456
english,20178
french,7445
german,3848
catalan,3370
...,...
scottish gaelix,1
serbian,1
swahili,1


In [ ]:
# Revisión

# Se ven valores únicos, teniendo en cuenta nulos, y cantidades
valores_unicos_nuevo_campo = registros[nuevo_campo].value_counts(dropna=False)

# Se ve cantidad de nulos en campo y no nulos:
valores_nulos_nuevo_campo = registros[nuevo_campo].isna().sum()
valores_nuevo_campo = registros[nuevo_campo].value_counts().sum()

# Se confirma que total registros compuestos por nulos y no nulos
assert len(registros) == valores_nulos_nuevo_campo + valores_nuevo_campo

# Se ven origen de todos los nulos en nuevo campo:
assert valores_nulos_nuevo_campo ==  registros[campo].isna().sum()

# Resumen:
total_grupo = registros[nuevo_campo].value_counts(dropna=False).reset_index()
total_grupo["%"] = ((total_grupo["count"] / len(registros))*100).round(2)
total_grupo = total_grupo.set_index(nuevo_campo)


print(f"Hay {valores_nuevo_campo} registros validos en {nuevo_campo}.\n")
print(f"Han aparecido {valores_nulos_nuevo_campo} valores nulos en {nuevo_campo}."
f"Los mismos que en {campo} : {registros[campo].isna().sum()}.\n"
f"\nSe han asignado como codigo corrupto "
f"{total_grupo.loc["codigo_corrupto",'count']} registros."
f"\nLos mismos que aparecian como no coincidentes {len(codigos_no_coincidente)}, "
f"suponiendo un {total_grupo.loc["codigo_corrupto",'%']}% de los registros.")

Hay 513743 registros validos en lengua_texto_008_descrito.

Han aparecido 1629 valores nulos en lengua_texto_008_descrito.Los mismos que en lengua_texto_008 : 1629.

Se han asignado como codigo corrupto 597 registros.
Los mismos que aparecian como no coincidentes 597, suponiendo un 0.12% de los registros.


### **4. df_dc4: cod_pais_marc -> pais_edicion_008**

In [ ]:
# Se establece variable para ir sustituyendo
soporte = df_dc4       #DF tabla soporte
campo = "pais_edicion_008"
nuevo_campo = campo + "_descrito"

In [ ]:
# Se comprueban valores únicos en código de tabla soporte
print(list(soporte['codigo'].unique()))

# Y nombre campos de tabla soporte
print(sorted(list(soporte.columns)))

['af', 'alu', 'aku', 'aa', 'abc', 'ae', 'as', 'an', 'ao', 'am', 'ay', 'aq', 'ag', 'azu', 'aru', 'ai', 'aw', 'at', 'aca', 'au', 'aj', 'bf', 'ba', 'bg', 'bb', 'bw', 'be', 'bh', 'dm', 'bm', 'bt', 'bo', 'bn', 'bs', 'bv', 'bl', 'bcc', 'bi', 'vb', 'bx', 'bu', 'uv', 'br', 'bd', 'cau', 'cb', 'cm', 'xxc', 'cv', 'ca', 'cj', 'cx', 'cd', 'cl', 'cc', 'ch', 'xa', 'xb', 'ck', 'cou', 'cq', 'cf', 'cg', 'ctu', 'cw', 'xga', 'cr', 'iv', 'ci', 'cu', 'co', 'cy', 'xr', 'deu', 'dk', 'dcu', 'ft', 'dq', 'dr', 'ec', 'ua', 'es', 'enk', 'eg', 'ea', 'er', 'sq', 'et', 'fk', 'fa', 'fj', 'fi', 'flu', 'fr', 'fg', 'fp', 'go', 'gm', 'gz', 'gau', 'gs', 'gw', 'gh', 'gi', 'gr', 'gl', 'gd', 'gp', 'gu', 'gt', 'gg', 'gv', 'pg', 'gy', 'ht', 'hiu', 'hm', 'ho', 'hu', 'ic', 'idu', 'ilu', 'ii', 'inu', 'io', 'iau', 'ir', 'iq', 'iy', 'ie', 'im', 'is', 'it', 'jm', 'ja', 'je', 'ji', 'jo', 'ksu', 'kz', 'kyu', 'ke', 'gb', 'kn', 'ko', 'kv', 'ku', 'kg', 'ls', 'lv', 'le', 'lo', 'lb', 'ly', 'lh', 'li', 'lau', 'lu', 'mg', 'meu', 'mw', 'my', '

In [ ]:
# Cambio tipo de dato a campo
registros[campo] = registros[campo].astype("category")

# Nulos en campo
print(f"Antes de la asignación hay {registros[campo].isna().sum()} valores nulos")

Antes de la asginación hay 10620 valores nulos


Se observa que hay 10.620 registros con valor nulo en código establecido para este campo, no constando la información en el catálogo.

Más adelante se mirará si es asignable de alguna manera.

In [ ]:
# Se comprueba que valores en campo de registros no están en los códigos de la tabla:

# Se establece filtro invertido -> se busca todos los registros que NO cumplen condición
sin_descripcion = ~registros[campo].isin(soporte["codigo"]) & ~registros[campo].isna()

# Se contempla Serie tratada en registros para los valores reales no encontrados
# en tabla soporte:
codigos_no_coincidente = registros.loc[sin_descripcion, campo]
print(f"Hay los siguientes registros según valor no coincidente: \n{codigos_no_coincidente.value_counts().sum()}"
" \n Cuyos valores son  distintos a los contemplados en tabla soporte.\n"
f"Los valores no coincidentes son: {codigos_no_coincidente.unique()}")


Hay los siguientes registros según valor no coincidente: 
3420 
 Cuyos valores son  distintos a los contemplados en tabla soporte.
Los valores no coincidentes son: ['xxx', 'xxr', 'ndl', 'mej', '#j', ..., 'fra', 'gbr', '-uk', 'xkk', 'mex']
Length: 190
Categories (308, string): [#//, #00, #b, #ca, ..., |sp, |||, ¿sp, ¿¿p]


Se observa que hay 3.420 valores presentes en el campo observado que no se encuentran en la tabla descriptiva.

Representan un pequeño % en la totalidad de registros. Se dejan como nulos por incorrecta codificación del catálogo, señalándose así en el nuevo campo, en el caso de necesitar estos registros,  se valorarán independientemente.



In [ ]:
# Sustitución de los códigos por la descripción:

# se establece el código como índice para hacer el cruce a partir de él y queda como
# único campo la descripción. Como diccionario
mapa = soporte.set_index("codigo")["nombre_en"]

registros[nuevo_campo] = registros[campo].map(mapa)

# Se añade comentario para los mal codificados:
registros.loc[sin_descripcion, nuevo_campo] = "codigo_corrupto"


# Cambio tipo de dato nuevo campo a category:
registros[nuevo_campo] = registros[nuevo_campo].astype("category")


In [ ]:
# Revisión

# Se ven valores únicos, teniendo en cuenta nulos, y cantidades
valores_unicos_nuevo_campo = registros[nuevo_campo].value_counts(dropna=False)

# Se ve cantidad de nulos en campo y no nulos:
valores_nulos_nuevo_campo = registros[nuevo_campo].isna().sum()
valores_nuevo_campo = registros[nuevo_campo].value_counts().sum()

# Se confirma que total registros compuestos por nulos y no nulos
assert len(registros) == valores_nulos_nuevo_campo + valores_nuevo_campo

# Se ven origen de todos los nulos en nuevo campo:
assert valores_nulos_nuevo_campo ==  registros[campo].isna().sum()

# Resumen:
total_grupo = registros[nuevo_campo].value_counts(dropna=False).reset_index()
total_grupo["%"] = ((total_grupo["count"] / len(registros))*100).round(2)
total_grupo = total_grupo.set_index(nuevo_campo)


print(f"Hay {valores_nuevo_campo} registros validos en {nuevo_campo}.\n")
print(f"Han aparecido {valores_nulos_nuevo_campo} valores nulos en {nuevo_campo}."
f"\nLos mismos que en {campo} : {registros[campo].isna().sum()}. "
f"Suponiendo un {total_grupo.iloc[1,1]}% de los registros. \n"
f"\nSe han asignado como codigo corrupto "
f"{total_grupo.loc["codigo_corrupto",'count']} registros."
f"\nLos mismos que aparecian como no coincidentes {len(codigos_no_coincidente)}, "
f"suponiendo un {total_grupo.loc["codigo_corrupto",'%']}% de los registros.")

Hay 504752 registros validos en pais_edicion_008_descrito.

Han aparecido 10620 valores nulos en pais_edicion_008_descrito.
Los mismos que en pais_edicion_008 : 10620. Suponiendo un 2.06% de los registros. 

Se han asignado como codigo corrupto 3420 registros.
Los mismos que aparecian como no coincidentes 3420, suponiendo un 0.66% de los registros.


### **5. df_dc5 : cod_publico_008 -> publico_008**

In [ ]:
# Se establece variable para ir sustituyendo

# DF tabla soporte
soporte = df_dc5

# Nombre campo

campo = "publico_008"

nuevo_campo = campo + "_descrito"

# Se comprueban valores únicos en código de tabla soporte
print(list(soporte['codigo'].unique()))

# Y nombre campos de tabla soporte
print(list(soporte.columns))

['#', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'j', '|']
['codigo', 'descripcion', 'fuente']


In [ ]:
# Cambio tipo de dato a campo
registros[campo] = registros[campo].astype("category")

# Nulos en campo
print(f"Antes de la asignación hay {registros[campo].isna().sum()} valores nulos")

Antes de la asginación hay 1203 valores nulos


Se observa que 1.203 registros no tienen código establecido para este campo, no constando la información en el catálogo. Estos, aparecerán como nulos en el nuevo campo ya que no hay referencia que asignar.

Más adelante se mirará si es asignable de alguna manera.

In [ ]:
# Se comprueba que valores en campo de registros no están en los códigos de la tabla:

# Se establece filtro invertido -> se busca todos los registros que NO cumplen condición
sin_descripcion = ~registros[campo].isin(soporte["codigo"]) & ~registros[campo].isna()

# Se contempla Serie tratada en registros para los valores reales no encontrados
# en tabla soporte:
codigos_no_coincidente = registros.loc[sin_descripcion, campo]
print(f"Hay los siguientes registros según valor no coincidente: \n{codigos_no_coincidente.value_counts().sum()}"
" \n Cuyos valores son  distintos a los contemplados en tabla soporte.\n"
f"Los valores no coincidentes son: {codigos_no_coincidente.unique()}")


Hay los siguientes registros según valor no coincidente: 
9919 
 Cuyos valores son  distintos a los contemplados en tabla soporte.
Los valores no coincidentes son: ['0', '/', 'y', 'p', 'i', ..., '2', 'l', 'h', '8', 'r']
Length: 19
Categories (29, string): [#, /, 0, 1, ..., s, u, y, |]


Se observa que hay 9.919 valores presentes en el campo observado que no se encuentran en la tabla descriptiva.

Representan un pequeño % en la totalidad de registros. Se dejan como nulos por incorrecta codificación del catálogo, señalándose así en el nuevo campo, en el caso de necesitar estos registros,  se valorarán independientemente.



In [ ]:
# Sustitución de los códigos por la descripción:

# se establece el código como índice para hacer el cruce a partir de él y queda como
# único campo la descripción. Como diccionario
mapa = soporte.set_index("codigo")["descripcion"]
registros[nuevo_campo] = registros[campo].map(mapa)

# Se añade comentario para los mal codificados:
registros.loc[sin_descripcion, nuevo_campo] = "codigo_corrupto"


# Cambio tipo de dato nuevo campo a category:
registros[nuevo_campo] = registros[nuevo_campo].astype("category")


In [ ]:
# Revisión

# Se ven valores únicos, teniendo en cuenta nulos, y cantidades
valores_unicos_nuevo_campo = registros[nuevo_campo].value_counts(dropna=False)

# Se ve cantidad de nulos en campo y no nulos:
valores_nulos_nuevo_campo = registros[nuevo_campo].isna().sum()
valores_nuevo_campo = registros[nuevo_campo].value_counts().sum()

# Se confirma que total registros compuestos por nulos y no nulos
assert len(registros) == valores_nulos_nuevo_campo + valores_nuevo_campo

# Se ven origen de todos los nulos en nuevo campo:
assert valores_nulos_nuevo_campo ==  registros[campo].isna().sum()

# Resumen:
total_grupo = registros[nuevo_campo].value_counts(dropna=False).reset_index()
total_grupo["%"] = ((total_grupo["count"] / len(registros))*100).round(2)
total_grupo = total_grupo.set_index(nuevo_campo)


print(f"Hay {valores_nuevo_campo} registros validos en {nuevo_campo}.\n")
print(f"Han aparecido {valores_nulos_nuevo_campo} valores nulos en {nuevo_campo}."
f"\nLos mismos que en {campo} : {registros[campo].isna().sum()}. "
f"Suponiendo un {total_grupo.iloc[-3,1]}% de los registros. \n"
f"\nSe han asignado como codigo corrupto "
f"{total_grupo.loc["codigo_corrupto",'count']} registros."
f"\nLos mismos que aparecian como no coincidentes {len(codigos_no_coincidente)}, "
f"suponiendo un {total_grupo.loc["codigo_corrupto",'%']}% de los registros.")
print(f"Además se observan {total_grupo.loc['no codificado', 'count']} registros con"
f" valor no codificado. Suponiendo el {total_grupo.loc['no codificado', '%']}%")

Hay 514169 registros validos en publico_008_descrito.

Han aparecido 1203 valores nulos en publico_008_descrito.
Los mismos que en publico_008 : 1203. Suponiendo un 0.23% de los registros. 

Se han asignado como codigo corrupto 9919 registros.
Los mismos que aparecian como no coincidentes 9919, suponiendo un 1.92% de los registros.
Además se observan 33473 registros con valor no codificado. Suponiendo el 6.49%


In [ ]:
total_grupo

,count,%
publico_008_descrito,,
desconocido o no especificado,287672,55.82
adulto,127202,24.68
no codificado,33473,6.49
juvenil,18406,3.57
primaria,14664,2.85
preescolar,14543,2.82
codigo_corrupto,9919,1.92
general,6001,1.16
preadolescente,1386,0.27


### **6. df_dc6 : cod_tipo_fecha_008 -> tipo_fecha_008**

In [ ]:
# Se establece variable para ir sustituyendo

# DF tabla soporte
soporte = df_dc6

# Nombre campo

campo = "tipo_fecha_008"

nuevo_campo = campo + "_descrito"

# Se comprueban valores únicos en código de tabla soporte
print(list(soporte['codigo'].unique()))

# Y nombre campos de tabla soporte
print(list(soporte.columns))

['b', 'c', 'd', 'e', 'i', 'k', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', '|']
['codigo', 'descripcion', 'fuente']


In [ ]:
# Cambio tipo de dato a campo
registros[campo] = registros[campo].astype("category")

# Nulos en campo
print(f"Antes de la asignación hay {registros[campo].isna().sum()} valores nulos")

Antes de la asginación hay 339 valores nulos


Se observa que 339 registros no tienen código establecido para este campo, no constando la información en el catálogo. Estos, aparecerán como nulos en el nuevo campo ya que no hay referencia que asignar.

Más adelante se mirará si es asignable de alguna manera.

In [ ]:
# Se comprueba que valores en campo de registros no están en los códigos de la tabla:

# Se establece filtro invertido -> se busca todos los registros que NO cumplen condición
sin_descripcion = ~registros[campo].isin(soporte["codigo"]) & ~registros[campo].isna()

# Se contempla Serie tratada en registros para los valores reales no encontrados
# en tabla soporte:
codigos_no_coincidente = registros.loc[sin_descripcion, campo]
print(f"Hay los siguientes registros según valor no coincidente: \n{codigos_no_coincidente.value_counts().sum()}"
" \n Cuyos valores son  distintos a los contemplados en tabla soporte.\n"
f"Los valores no coincidentes son: {codigos_no_coincidente.unique()}")


Hay los siguientes registros según valor no coincidente: 
13373 
 Cuyos valores son  distintos a los contemplados en tabla soporte.
Los valores no coincidentes son: ['#', '1', '2', '0', '8', ..., '[', 'g', '~', 'j', 'º']
Length: 19
Categories (34, string): [#, ', +, 0, ..., u, |, ~, º]


In [ ]:
# Se ve cantidad de valores #

valor_des_ni = codigos_no_coincidente.value_counts().loc["#"]
print(f"Hay {valor_des_ni} correspondientes a 'blanco - no valido (#)'")
print(f"Dejando {codigos_no_coincidente.value_counts().sum() - valor_des_ni}"
" pendientes de determinar.")


Hay 12854 correspondientes a 'blanco - no valido (#)'
Dejando 519 pendientes de determinar.


Se observa que hay 13.373 valores presentes en el campo observado que no se encuentran en la tabla descriptiva. De los cuales, 12.854 corresponden al valor en blanco no válido. Se señalan como tal.

Dejando 519 valores pendiente de determinar o corruptos. Representan un pequeño % en la totalidad de registros. Se dejan como nulos por incorrecta codificación del catálogo, señalándose así en el nuevo campo, en el caso de necesitar estos registros,  se valorarán independientemente.


In [ ]:
# Sustitución de los códigos por la descripción:

# se establece el código como índice para hacer el cruce a partir de él y queda como
# único campo la descripción. Como diccionario
mapa = soporte.set_index("codigo")["descripcion"]
mapa.loc["#"] = "blanco(no valido)"
registros[nuevo_campo] = registros[campo].map(mapa)

# Se añade comentario para los mal codificados:
blanco = sin_descripcion & ~registros[campo].isin(["#"])
registros.loc[blanco, nuevo_campo] = "codigo_corrupto"


# Cambio tipo de dato nuevo campo a category:
registros[nuevo_campo] = registros[nuevo_campo].astype("category")


In [ ]:
# Revisión

# Se ven valores únicos, teniendo en cuenta nulos, y cantidades
valores_unicos_nuevo_campo = registros[nuevo_campo].value_counts(dropna=False)

# Se ve cantidad de nulos en campo y no nulos:
valores_nulos_nuevo_campo = registros[nuevo_campo].isna().sum()
valores_nuevo_campo = registros[nuevo_campo].value_counts().sum()

# Se confirma que total registros compuestos por nulos y no nulos
assert len(registros) == valores_nulos_nuevo_campo + valores_nuevo_campo

# Se ven origen de todos los nulos en nuevo campo:
assert valores_nulos_nuevo_campo ==  registros[campo].isna().sum()

# Resumen:
total_grupo = registros[nuevo_campo].value_counts(dropna=False).reset_index()
total_grupo["%"] = ((total_grupo["count"] / len(registros))*100).round(2)
total_grupo = total_grupo.set_index(nuevo_campo)


print(f"Hay {valores_nuevo_campo} registros validos en {nuevo_campo}.\n")
print(f"Han aparecido {valores_nulos_nuevo_campo} valores nulos en {nuevo_campo}."
f"\nLos mismos que en {campo} : {registros[campo].isna().sum()}. "
f"Suponiendo un {total_grupo.iloc[-8,1]}% de los registros. \n"
f"\n- Han aparecido {total_grupo.loc['blanco(no valido)', 'count']} registros"
f" en blanco no válidos(#), \nsuponiendo un "
f"{total_grupo.loc['blanco(no valido)', '%']}  % de los registros.\n"
f"\n- Se han asignado como codigo corrupto."
f"{total_grupo.loc["codigo_corrupto",'count']} registros."
f"\n  Los mismos que aparecian como no coincidentes {len(codigos_no_coincidente)-valor_des_ni}, "
f"suponiendo\n un {total_grupo.loc["codigo_corrupto",'%']}% de los registros.\n")
print(f"- Se observan {total_grupo.loc['no codificado', 'count']} registros con"
f" valor no codificado. Suponiendo el {total_grupo.loc['no codificado', '%']}%")

Hay 515033 registros validos en tipo_fecha_008_descrito.

Han aparecido 339 valores nulos en tipo_fecha_008_descrito.
Los mismos que en tipo_fecha_008 : 339. Suponiendo un 0.07% de los registros. 

- Han aparecido 12854 registros en blanco no válidos(#), 
suponiendo un 2.49  % de los registros.

- Se han asignado como codigo corrupto.519 registros.
  Los mismos que aparecian como no coincidentes 519, suponiendo
 un 0.1% de los registros.

- Se observan 175 registros con valor no codificado. Suponiendo el 0.03%


In [ ]:
total_grupo

,count,%
tipo_fecha_008_descrito,,
fecha unica conocida o probable (fecha2 en blanco),492195,95.50
blanco(no valido),12854,2.49
fechas multiples (fecha1 = inicio; fecha2 = fin),3113,0.60
reimpresion/reedicion (fecha1 = reproduccion; fecha2 = fecha de la original),2403,0.47
recurso continuo en curso (fecha1 = inicio; fecha2 = 9999),1054,0.20
fecha dudosa (fecha1 = mas temprana; fecha2 = mas tardia),811,0.16
fecha detallada (fecha1 = ano; fecha2 = mes y dia),616,0.12
fechas desconocidas,539,0.10
codigo_corrupto,519,0.10


### Eliminar campos códigos

En caso de necesitar consulta se observarán en capa bronze.

In [ ]:
campos_eliminar = ['tipo_fecha_008', 'publico_008', 'pais_edicion_008',
                   'lengua_texto_008', 'forma_literaria_008',
                   'forma_literaria_cdu']
# Se comprueba que la lista de campos a eliminar contenga solo valores únicos.
# Y que después de comprobarlo tenga la misma cantidad de valores.
assert len(campos_eliminar) == len(set(campos_eliminar))
print(f"Columnas iniciales: {registros.shape[1]}."
f"\nColumnas a eliminar: {len(campos_eliminar)}")
registros = registros.drop(columns=campos_eliminar, errors="ignore")
print(f"Columnas finales : {len(registros.columns)}")

del campos_eliminar

Columnas iniciales: 26.
Columnas a eliminar: 6
Columnas finales : 20


### Revisión composición campo titulo



In [ ]:
# Se miran que símbolos no alfanuméricos hay, excluyendo la coma:
titulo= registros['titulo_245a']
simbolos = titulo.str.findall(r'[^a-zA-Z0-9\s]').explode().value_counts()
print(simbolos.reset_index()['titulo_245a'].unique())

# Se busca qué filas con valores en título:
con_simbolos = titulo.str.contains(r'[^a-zA-Z0-9\s]', na=False)

reg_simbolos= registros.loc[con_simbolos, 'titulo_245a']
print(len(reg_simbolos))
display(reg_simbolos.head(5))

['*' ',' '.' '-' '"' '!' '(' ')' '¡' '?' "'" '¿' ':' ';' '_' '/' '[' ']'
 '&' 'º' '´' '=' '+' '#' 'ª' '@' '®' 'þ' '`' '%' '$' '·' 'ð' '\xad' 'ß'
 '\x80' 'ø' '>' '{' '©' '°' '<' '^' '²' '£' '\x94' '\x9c' '\\' '\x93' '¸'
 '¬' '}' '\x81' 'æ' '¹' '\x9e' '\x97']
92161


,titulo_245a
0,"ley 29/1994, de 24 de noviembre de arrendamien..."
19,"ley de medidas de politica territorial, suelo ..."
36,"ley de medidas de politica territorial, suelo ..."
45,reglamento de procedimiento en las reclamacion...
53,"abogados, procuradores y funcionarios de la ad..."


Con tal de normalizar los valores lo máximo posible para poder comparar con prestamos, de ser necesario. Se eliminan todos los símbolos y signos de puntuación encontrados. ['*', ',', '.', '-', '"','!', '(', ')',
 '¡',
 '?',
 "'",
 '¿',
 ':',
 ';',
 '_',
 '/',
 '[',
 ']',
 '&',
 'º',
 '´',
 '=',
 '+',
 '#',
 'ª',
 '@',
 '®',
 'þ',
 '`',
 '%',
 '$',
 '·',
 'ð',
 '\xad',
 'ß',
 '\x80',
 'ø',
 '>',
 '{',
 '©',
 '°',
 '<',
 '^',
 '²',
 '£',
 '\x94',
 '\x9c',
 '\\', '\x93','¸', '¬', '}', '\x81', 'æ', '¹','\x9e','\x97'].


Encontrandose en 92.161 registros.

Entre los valores incorrectos se ha detectado valores por error de codificación \x94','\x9c','\x93','\x81', '\x9e','\x97', '\x80' se vuelven a decodificar para que salgan bien y tratarlos si necesario

Se mantienen los números, se quitan comas.

In [ ]:
# Decodificación correcta:

# Número y visualización de los valores:
mask_cod = titulo.str.contains(r'[\x80-\x9f]', na=False)
print(mask_cod.sum())
display(titulo[mask_cod].head(5))

#Decodificación

def reparar(t):
    try:
        return t.encode('latin-1').decode('cp1252')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return t

registros.loc[mask_cod, 'titulo_245a'] = titulo[mask_cod].map(reparar)

#Comprobación
registros['titulo_245a'].str.contains(r'[\x80-\x9f]', na=False).sum()

17


,titulo_245a
185376,hansel y gretel  humperdinck / garcia maurino
236461,comer bien en familia por 9  al dia
291289,100 startup
317626,recetas por 5 
339106,1 


np.int64(1)

In [ ]:
# Eliminación de símbolos:

sin_espacio = r"[\xad·'´`ºª°¹²®©¸]"

registros['titul_norm'] = (
    registros['titulo_245a']
    .str.replace(sin_espacio, '', regex=True)
    .str.replace(r'[\x80-\x9f]', ' ', regex=True) #Por los que no se pudieron borrar
    .str.replace(r'[^\w\s]|[\d_]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

# Comprobación

registros['titul_norm'].str.contains(r'[^a-zA-Z0-9\s]', na=False).sum()

np.int64(0)

Durante la comprobacion se ha visto que hay titulos que presentan espacios tanto al inicio como al final, derivadas de la normalizacion.

Se quitan tanto para titulo como para autor en registros y prestamos.


In [ ]:
registros["titulo"] = registros["titulo"].str.split().str.join(" ")
registros["autor"] = registros["autor"].str.split().str.join(" ")

In [ ]:
# Comprobacion
t = registros["titulo"].dropna()

print((t != t.str.strip()).sum())                        # espacios al inicio o al final
print(t.str.contains("[\u200b-\u200d\u2060\ufeff]").sum())  # caracteres invisibles

### Revisión composición campo ISBN

In [ ]:
# Se mira que registros ISBN sean solo números:

isbn = registros['isbn_original']

# Se mira que símbolos no numéricos hay:

# Contemplando si registros sin números:
no_num = isbn.notna() &  ~isbn.str.contains(r'\d', na=False)

# Se miran registros de haberlo
sin_numeros = registros.loc[no_num, 'isbn_original']
print(len(sin_numeros))
sin_numeros.unique()

2


<ArrowStringArray>
['en tramite', 'texto en frances']
Length: 2, dtype: string

Se han encontrado 2 registros que no contienen números, se pasa a nulo

In [ ]:
# Se pasan estos registros a nulos por incorrecta decodificación:

print(f"Nulos campos iniciales {registros['isbn_original'].isna().sum()}")
registros.loc[no_num, 'isbn_original'] = pd.NA
print(f"Nulos campos finales {registros['isbn_original'].isna().sum()}")

Nulos campos iniciales 44657
Nulos campos finales 44659


In [ ]:
# Se buscan símbolos no numéricos en isbn:

no_numeros = isbn.str.findall(r'[^0-9\s]').explode().value_counts()
print(no_numeros.index)

# Se busca qué filas con valores en autor:
mask_isbn = isbn.str.contains(r'[^0-9\s]', na=False)

reg_numeros_isbn= registros.loc[mask_isbn, 'isbn_original']
print(len(reg_numeros_isbn))
display(reg_numeros_isbn.head(5))

Index(['-', '|', 'x', 'o', 'e', 'r', 'i', '.', 'n', 'a', 't', 'c', '(', ')',
       'l', 's', 'u', 'd', 'm', 'b', 'v', 'g', 'p', ']', '[', '/', 'k', ',',
       ':', 'f', 'h', '"', '+', 'y', '_', '=', 'j', 'z', '', 'w', '?', 'º',
       ''', '#', '&', 'q', '$', '´', ';'],
      dtype='object', name='isbn_original')
464130


,isbn_original
0,84-340-0742-8
1,84-7491-537-6
2,84-7357-480-x
3,84-7357-769-8
4,84-470-0480-5


Se han encontrado las siguientes letras y símbolos en el campo ISBN que debería ser exclusivamente numérico, se extrae también el guion que separa algunos registros para normalizarlos todos entre ellos: ['-',
 'x',
 'o',
 'e',
 'r',
 'i',
 '.',
 'n',
 'a',
 't',
 'c',
 '(',
 ')',
 'l',
 's',
 'u',
 'd',
 'm',
 'b',
 'v',
 'g',
 'p',
 ']',
 '[',
 '/',
 'k',
 ',',
 ':',
 'f',
 'h',
 '"',
 '+',
 'y',
 '_',
 '=',
 'j',
 'z',
 '\x80',
 'w',
 '?',
 'º',
 "'",
 '#',
 '&',
 'q',
 '$',
 '´',
 ';'] Repartidos en 463.130 registros.

 Se ha excluido de la lista el signo or (|) ya que señala la existencia de 2 números ISBN.

In [ ]:
# Eliminación símbolos, letras del campo isbn:
registros['isbn_limpio'] = (
    registros['isbn_original']
    .astype('string')
    .str.replace(r'[^0-9|]', '', regex=True)  # deja solo dígitos y |
    .str.replace(r'\|+', '|', regex=True)     # colapsa || seguidos
    .str.strip('|')                           # quita | al inicio/final
    .replace('', pd.NA)                       # lo que queda vacío → nulo
)
# Comprobación
registros['isbn_limpio'].str.contains(r'[^0-9|\s]', na=False).sum()

np.int64(0)

In [ ]:
# Comprobación tamaño código ISBN:

# Longitud de los códigos :

registros['isbn_longitud'] = registros['isbn_limpio'].str.len()

display(registros['isbn_longitud'].value_counts(normalize=True, dropna= False).sort_index()*100)

,proportion
isbn_longitud,
2,0.000582
5,0.000582
6,0.00097
7,0.004463
8,0.016299
...,...
867,0.000194
909,0.000194
1105,0.000194


Los ISBN con longitudes largas pertenecen a registros que unifican bajo un mismo registro diversos ejemplares o tomos. Extrapolación ya que se puede ver que el registro con una longitud de ISBN de 1581 corresponde al título One Piece.

### Revisión composición campo autor

In [ ]:
# Se miran que símbolos no alfanuméricos hay, excluyendo la coma:
autor= registros['autor']
numeros = autor.str.findall(r'[^a-zA-Z0-9\s,]').explode().value_counts()
print(numeros.reset_index()['autor'].unique())

# Se busca qué filas con valores en autor:
con_numeros = autor.str.contains(r'[^a-zA-Z0-9\s,]', na=False)

reg_numeros= registros.loc[con_numeros, 'autor']
print(len(reg_numeros))
display(reg_numeros.head(5))

['.' '-' "'" '(' ')' '"' 'ø' '&' 'ª' ';' '#' '@' '´' 'ð' 'º' '·' ':' '/'
 '\x9a' '`' '_' '$' '\x8a' '!' 'ß' '\x9c' '?' '\x8e' 'æ' '°' '²' '+'
 '\x92' '¡']
36535


,autor
4,madrid (comunidad autonoma)
8,madrid (comunidad autonoma)
19,madrid (comunidad autonoma)
21,"sanchez-arcilla, jose"
36,madrid (comunidad autonoma)


Se ven los siguientes símbolos:

['.' '-' "'" '(' ')' '"' 'ø' '&' 'ª' ';' '#' '@' '´' 'ð' 'º' '·' ':' '/'
 '\x9a' '`' '_' '$' '\x8a' '!' 'ß' '\x9c' '?' '\x8e' 'æ' '°' '²' '+'
 '\x92' '¡'] a lo largo de 36.535 registros.

 Al igual que en el título, se observan errores de decodificación, se vuelven a decodificar.
 Se procede a eliminarlos.

In [ ]:
# Decodificación correcta:

# Número y visualización de los valores:
con_numeros = autor.str.contains(r'[\x80-\x9f]', na=False)
print(con_numeros.sum())
display(autor[con_numeros].head(5))

#Decodificación

def reparar(t):
    try:
        return t.encode('latin-1').decode('cp1252')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return t

registros.loc[con_numeros, 'autor'] = autor[con_numeros].map(reparar)

#Comprobación
registros['autor'].str.contains(r'[\x80-\x9f]', na=False).sum()

15


,autor
341432,"acek, jiri"
448604,"omara, s. m."
465593,"filipi, toma"
478116,"vankmajer, jan"
480538,"haniova, viktorie"


np.int64(0)

In [ ]:
# Eliminación caracteres y números años de nacimiento y muerte

sin_espacio = r"[\xad·'´`ºª°¹²®©]"

registros['autor_norm'] = (
    registros['autor']
    .str.replace(r'\s*\([^)]*\)', '', regex=True)
    .str.replace(sin_espacio, '', regex=True)
    .str.replace(r'[^\w\s,]|[\d_]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(r'\s*,\s*', ', ', regex=True)
    .str.replace(r',(\s*,)+', ',', regex=True)
    .str.strip(' ,'))

# Comprobación
registros['autor_norm'].str.contains(r'[^a-zA-Z0-9\s,]', na=False).sum()

np.int64(0)

### Revisión campo tipo fecha

 anio_publicacion y fecha1_008. Ambas señalan el año de publicación de la edición representada en el registro. Pero se extraen de lugares distintos dentro del catálogo.

 - fecha1_008 se extrae del código 008 (datos codificados). La fecha presente en este campo depende del tipo de fecha señalado dentro del mismo código 008, en la tabla se muestra en el campo tipo_de_fecha.

 - anio_publicacion se extrae de la cabecera del registro y es una anotación libre del catalogador, texto libre.

 El código 008 es un código

In [ ]:
# Comparación entre anio_publicacion y fecha1_008
# para ver coincidencias y diferencias.

# Pasar las dos fechas a texto y los nulos a "".
# Y poner los nulos en campo vacio
# Se ha de comparar con mismos tipos
d = registros[["id_registro", "fecha1_008", "anio_publicacion", "tipo_fecha_008_descrito"]]
for c in ["fecha1_008", "anio_publicacion", "tipo_fecha_008_descrito"]:
    d[c] = d[c].astype('string').fillna("")

f008_ok = d["fecha1_008"].str.fullmatch(r"\d{4}")   # año de 4 dígitos
f260_ok = d["anio_publicacion"] != ""

condiciones = [
    ~f008_ok & ~f260_ok,
    ~f008_ok &  f260_ok,
     f008_ok & ~f260_ok,
     d["fecha1_008"] == d["anio_publicacion"],
]
etiquetas = ["ninguna", "solo_260", "solo_008", "coinciden"]
d["estado"] = np.select(condiciones, etiquetas, default="difieren")

print(len(d))
display(d["estado"].value_counts(normalize=True).round(3))
del d

515372


,proportion
estado,
coinciden,0.929
solo_008,0.046
ninguna,0.014
difieren,0.008
solo_260,0.002


In [ ]:
# --- 1. Pasar las dos fuentes a número ----------------------------------------
# errors="coerce": todo lo que no sea un número ("####", "19uu", "|||", vacío/NA)
# se convierte en nulo. Así solo quedan años numéricos que se pueden validar.
f008 = pd.to_numeric(registros["fecha1_008"], errors="coerce")
f260 = pd.to_numeric(registros["anio_publicacion"], errors="coerce")

# --- 2. Validez: año entre 1500 y 2026 (el catálogo se descargó en 2026) ------
ANIO_MIN, ANIO_MAX = 1500, 2026
ok_008 = f008.between(ANIO_MIN, ANIO_MAX).fillna(False).astype(bool) #tipo numpy para comparar
ok_260 = f260.between(ANIO_MIN, ANIO_MAX).fillna(False).astype(bool) #tipo numpy para comparar

# --- 3. Prioridad: primero el 008; si no es válido, el 260; si no, nulo -------
# np.select evalúa las condiciones en orden y aplica la primera que se cumple.
v008 = f008.to_numpy(dtype=float, na_value=np.nan)
v260 = f260.to_numpy(dtype=float, na_value=np.nan)

registros["anio_edicion"] = pd.Series(
    np.select([ok_008, ok_260], [v008, v260], default=np.nan),
    index=registros.index,
).astype("Int64")                            # entero que admite nulos (<NA>)

registros["fuente_anio"] = np.select([ok_008, ok_260], ["008", "260"], default="ninguna")

# --- 4. Controles para el registro de limpieza --------------------------------
print("Libros:", len(registros))
print(registros["fuente_anio"].value_counts())
print(registros["fuente_anio"].value_counts(normalize=True).round(4))

# Años imposibles descartados en cada fuente (numéricos, pero fuera de 1500-2026)
print("\n008 fuera de rango:", (f008.notna() & ~ok_008).sum())
print("260 fuera de rango:", (f260.notna() & ~ok_260).sum())

# Comprobación de que el resultado queda dentro del intervalo
print(registros["anio_edicion"].min(), registros["anio_edicion"].max())

Libros: 515372
fuente_anio
008        507036
ninguna      7414
260           922
Name: count, dtype: int64
fuente_anio
008        0.9838
ninguna    0.0144
260        0.0018
Name: proportion, dtype: float64

008 fuera de rango: 39
260 fuera de rango: 0
1552 2026


In [ ]:
registros.columns

Index(['id_registro', 'len_008', 'fecha1_008', 'traductor_700',
       'mencion_traduccion_245c', 'titulo_245a', 'autor', 'isbn_original',
       'anio_publicacion', 'cdu_080', 'genero_literario_cdu', 'materias_650',
       'generos_655', 'n_ejemplares', 'forma_literaria_cdu_descrito',
       'forma_literaria_008_descrito', 'lengua_texto_008_descrito',
       'pais_edicion_008_descrito', 'publico_008_descrito',
       'tipo_fecha_008_descrito', 'titul_norm', 'autor_norm', 'isbn_limpio',
       'isbn_longitud', 'anio_edicion', 'fuente_anio'],
      dtype='object')

In [ ]:
columnas_ok ={
    'id_registro':'id_registro',
    'titul_norm':'titulo',
    'autor_norm':'autor',
    'anio_edicion': 'anio_edicion',
    'pais_edicion_008_descrito':'pais_edicion',
    'generos_655': 'genero',
    'publico_008_descrito': 'tipo_publico',
    'isbn_limpio': 'isbn',
    'n_ejemplares':'n_ejemplares'
    }

registros = registros[list(columnas_ok)].rename(columns=columnas_ok)

In [ ]:
anio = registros["anio_edicion"].astype("Int64")
print(anio.describe())
print(anio[(anio < 1800) | (anio > 2026)].value_counts().sort_index())

count       507958.0
mean     2003.786849
std        15.281689
min           1552.0
25%           1997.0
50%           2007.0
75%           2014.0
max           2026.0
Name: anio_edicion, dtype: Float64
anio_edicion
1552    2
1566    1
1574    1
1582    2
1584    1
       ..
1792    2
1795    1
1796    2
1797    2
1798    1
Name: count, Length: 67, dtype: Int64


In [ ]:
# Asignación n_ejemplares y año_edicion a Int64
registros['n_ejemplares'] = registros['n_ejemplares'].astype('Int64')
registros["anio_edicion"] = registros["anio_edicion"].astype("Int64")

In [ ]:
print(registros['anio_edicion'].min(), registros['anio_edicion'].max())

1552 2026


### valores únicos en tipo público


unifico valores con los presentes en prestamos y englobo los códigos sin información ya sea por desconocido, no informado o por no codificado.

Mantengo la distinción de primaria hasta ver como se catalogan en prcoc1. No encajan ni en juvenil ni en prelectores.

In [ ]:
registros['tipo_publico'].value_counts()

#Unifico valores con los presentes en prestamos
publico = {
    'desconocido o no especificado':'desconocido',
    'adulto': 'adultos',
    'no codificado': 'desconocido',
    'juvenil': 'juvenil',
    'primaria': 'primaria',
    'preescolar':'preescolar',
    'codigo_corrupto':'desconocido',
    'general': 'general',
    'preadolescente': 'juvenil',
    'adolescente': 'juvenil',
    'especializado':'especializado'
    }

# Paso a texto, reclasifico nombre y vuelvo a pasar a category

orig = registros["tipo_publico"].astype("string")
registros["tipo_publico"] = orig.map(publico).astype("category")

### Exportación parquet

In [ ]:
registros.to_parquet(
    DRIVE_DESTINO / 'registros.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)